# v11 repaired - DPO arms and paired statistics

HaNoRec (pinned `587face7`) CF-hardness reranker over frozen LLM2Rec + SASRec top-20 on Amazon Games.
This notebook is generated by `scripts/build_hanorec_notebook.py`; sources below are embedded and hash-checked.

**Repairs vs the exploratory run:** SFT now learns the binary Yes/No task (50% Yes / 50% No, as upstream);
DPO policy runs with dropout off (LLaMA-Factory default); LoRA is language-model only (vision tower frozen);
equal scores keep the retriever order. **Adaptations are recorded, not hidden:** Kaggle serves a T4 (bf16 emulated),
so `max_pixels` and epoch counts are derived from measured speed/VRAM and written to the job manifest.
**Not a confirmation:** one seed, validation only, test never scored.

## 1. Environment and GPU gate

In [ ]:
import hashlib, json, os, subprocess, sys, time
from pathlib import Path

NOTEBOOK_T0 = time.monotonic()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0", "pillow==11.0.0"], check=True)
import torch, transformers, peft

assert torch.cuda.is_available(), "GPU required"
GPU = {
    "name": torch.cuda.get_device_name(0),
    "total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "capability": list(torch.cuda.get_device_capability(0)),
    "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
}
# torch.cuda.is_bf16_supported() is True on T4 by emulation, so it proves nothing. Kaggle's kernel API only
# offers T4/P100 (NvidiaL4 is silently served as a T4), so bf16 is emulated: recorded as an adaptation.
GPU["bf16_native"] = GPU["capability"][0] >= 8
print(GPU)
assert GPU["name"] and torch.cuda.is_bf16_supported(), "bf16 (even emulated) is required by the protocol"

## 2. Embedded, hash-verified sources

In [ ]:
EMBEDDED_SOURCES = {"prep.py": "...", "train.py": "...", "analysis.py": "IiIiVmFsaWRhdGlvbi1vbmx5IHBhaXJlZCBzdGF0aXN0aWNzIGZvciB0aGUgcmVwYWlyZWQgdjExIGdhdGUgKHN0YW5kYXJkIGxpYnJhcnkgb25seSkuCgpNZXRyaWNzIGFyZSBhbHdheXMgY29tcHV0ZWQgb24gdGhlICpmdWxsKiB2YWxpZGF0aW9uIGNvaG9ydDogdXNlcnMgd2hvc2UgdGFyZ2V0IGlzCmFic2VudCBmcm9tIFNBU1JlYydzIHRvcC0yMCBjb250cmlidXRlIGV4YWN0bHkgMCB0byBldmVyeSByZXJhbmtlciAoYW5kIHRvIHRoZQpTQVNSZWMgb3JkZXIpLCBzbyBza2lwcGluZyB0aGUgUXdlbiBzY29yaW5nIGZvciB0aGVtIGNoYW5nZXMgbm8gbnVtYmVyLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBJdGVyYWJsZQoKTUVUUklDUyA9ICgibmRjZ0AxMCIsICJyZWNhbGxAMTAiKQpNSU5fTUVBTklOR0ZVTF9ORENHX0RFTFRBID0gMC4wMDIgICMgUGhhc2UgMSBmcm96ZW4gdGhyZXNob2xkIChwbGFucy8yNjA5MjAtKi9waGFzZS0wMS1zdGFydC5tZCkKCgpkZWYgX2tleShyb3c6IGRpY3Rbc3RyLCBBbnldKSAtPiB0dXBsZVtzdHIsIGludF06CiAgICByZXR1cm4gKHN0cihyb3dbInVzZXJfaWQiXSksIGludChyb3dbInRhcmdldCJdKSkKCgpkZWYgZnVsbF9jb2hvcnRfdmFsdWVzKGNvaG9ydDogbGlzdFtkaWN0W3N0ciwgQW55XV0sIHByZWRpY3Rpb25zOiBJdGVyYWJsZVtkaWN0W3N0ciwgQW55XV0pIC0+IGxpc3RbZGljdFtzdHIsIGZsb2F0XV06CiAgICAiIiJFeHBhbmQgaW5mb3JtYXRpdmUtdXNlciBwcmVkaWN0aW9ucyB0byB0aGUgZnVsbCBjb2hvcnQ7IGFic2VudCB1c2VycyBzY29yZSAwLiIiIgogICAgYnlfa2V5ID0ge19rZXkocm93KTogcm93IGZvciByb3cgaW4gcHJlZGljdGlvbnN9CiAgICB1bmtub3duID0gc2V0KGJ5X2tleSkgLSB7X2tleShyb3cpIGZvciByb3cgaW4gY29ob3J0fQogICAgaWYgdW5rbm93bjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie2xlbih1bmtub3duKX0gcHJlZGljdGlvbnMgYXJlIG91dHNpZGUgdGhlIHZhbGlkYXRpb24gY29ob3J0IikKICAgIHNraXBwZWRfaW5mb3JtYXRpdmUgPSBbCiAgICAgICAgcm93IGZvciByb3cgaW4gY29ob3J0CiAgICAgICAgaWYgX2tleShyb3cpIG5vdCBpbiBieV9rZXkgYW5kIGZsb2F0KHJvdy5nZXQoImNhbmRpZGF0ZV9yZWNhbGxAMjAiLCAwLjApKSA9PSAxLjAKICAgIF0KICAgIGlmIHNraXBwZWRfaW5mb3JtYXRpdmU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntsZW4oc2tpcHBlZF9pbmZvcm1hdGl2ZSl9IGluZm9ybWF0aXZlIHVzZXJzICh0YXJnZXQgaW4gdG9wLTIwKSBoYXZlIG5vIHByZWRpY3Rpb24iKQogICAgemVybyA9IHsibmRjZ0AxMCI6IDAuMCwgInJlY2FsbEAxMCI6IDAuMCwgImNhbmRpZGF0ZV9yZWNhbGxAMjAiOiAwLjB9CiAgICByZXR1cm4gWwogICAgICAgIHttZXRyaWM6IGZsb2F0KGJ5X2tleS5nZXQoX2tleShyb3cpLCB6ZXJvKVttZXRyaWNdKSBmb3IgbWV0cmljIGluICgqTUVUUklDUywgImNhbmRpZGF0ZV9yZWNhbGxAMjAiKX0KICAgICAgICBmb3Igcm93IGluIGNvaG9ydAogICAgXQoKCmRlZiBzdW1tYXJpemUodmFsdWVzOiBsaXN0W2RpY3Rbc3RyLCBmbG9hdF1dKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgaWYgbm90IHZhbHVlczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlbXB0eSBjb2hvcnQiKQogICAgcmV0dXJuIHttZXRyaWM6IHN1bSh2W21ldHJpY10gZm9yIHYgaW4gdmFsdWVzKSAvIGxlbih2YWx1ZXMpIGZvciBtZXRyaWMgaW4gKCpNRVRSSUNTLCAiY2FuZGlkYXRlX3JlY2FsbEAyMCIpfQoKCmRlZiBwYWlyZWRfYm9vdHN0cmFwKAogICAgYTogbGlzdFtkaWN0W3N0ciwgZmxvYXRdXSwKICAgIGI6IGxpc3RbZGljdFtzdHIsIGZsb2F0XV0sCiAgICBtZXRyaWM6IHN0ciA9ICJuZGNnQDEwIiwKICAgIG46IGludCA9IDIwMDAwLAogICAgc2VlZDogaW50ID0gMjAyNjEwMDEsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJQYWlyZWQgdXNlci1sZXZlbCBib290c3RyYXAgb2YgbWVhbihhIC0gYik7IENJIGlzIHRoZSAyLjUtOTcuNSBwZXJjZW50aWxlIGludGVydmFsLiIiIgogICAgaWYgbGVuKGEpICE9IGxlbihiKSBvciBub3QgYToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwYWlyZWQgY29ob3J0cyBtdXN0IGJlIGVxdWFsLWxlbmd0aCBhbmQgbm9uLWVtcHR5IikKICAgIGRlbHRhcyA9IFtmbG9hdCh4W21ldHJpY10pIC0gZmxvYXQoeVttZXRyaWNdKSBmb3IgeCwgeSBpbiB6aXAoYSwgYildCiAgICBzaXplID0gbGVuKGRlbHRhcykKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG1lYW5zID0gc29ydGVkKHN1bShkZWx0YXNbcm5nLnJhbmRyYW5nZShzaXplKV0gZm9yIF8gaW4gcmFuZ2Uoc2l6ZSkpIC8gc2l6ZSBmb3IgXyBpbiByYW5nZShuKSkKICAgIG1lYW5fZGVsdGEgPSBzdW0oZGVsdGFzKSAvIHNpemUKICAgICMgTW9udGUtQ2FybG8gcC12YWx1ZSB3aXRoIHRoZSArMSBjb3JyZWN0aW9uOiBuZXZlciBleGFjdGx5IDAgZnJvbSBhIGZpbml0ZSBudW1iZXIgb2YgcmVzYW1wbGVzLgogICAgcF9sZSA9IChzdW0obSA8PSAwIGZvciBtIGluIG1lYW5zKSArIDEpIC8gKG4gKyAxKQogICAgcF9nZSA9IChzdW0obSA+PSAwIGZvciBtIGluIG1lYW5zKSArIDEpIC8gKG4gKyAxKQogICAgcmV0dXJuIHsKICAgICAgICAibWV0cmljIjogbWV0cmljLAogICAgICAgICJ1c2VycyI6IHNpemUsCiAgICAgICAgIm1lYW5fZGVsdGEiOiBtZWFuX2RlbHRhLAogICAgICAgICJjaTk1IjogW21lYW5zW2ludCgwLjAyNSAqIG4pXSwgbWVhbnNbbWluKG4gLSAxLCBpbnQoMC45NzUgKiBuKSldXSwKICAgICAgICAicF90d29fc2lkZWQiOiBtaW4oMS4wLCAyICogbWluKHBfbGUsIHBfZ2UpKSwKICAgICAgICAiY2hhbmdlZF91c2VycyI6IHN1bShkICE9IDAgZm9yIGQgaW4gZGVsdGFzKSwKICAgICAgICAicG9zaXRpdmVfdXNlcnMiOiBzdW0oZCA+IDAgZm9yIGQgaW4gZGVsdGFzKSwKICAgICAgICAibmVnYXRpdmVfdXNlcnMiOiBzdW0oZCA8IDAgZm9yIGQgaW4gZGVsdGFzKSwKICAgIH0KCgpkZWYgaG9sbV9hZGp1c3QocF92YWx1ZXM6IGRpY3Rbc3RyLCBmbG9hdF0pIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiJIb2xtIHN0ZXAtZG93biBhZGp1c3RlZCBwLXZhbHVlcy4iIiIKICAgIG9yZGVyZWQgPSBzb3J0ZWQocF92YWx1ZXMuaXRlbXMoKSwga2V5PWxhbWJkYSBpdGVtOiBpdGVtWzFdKQogICAgYWRqdXN0ZWQ6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7fQogICAgcnVubmluZyA9IDAuMAogICAgZm9yIHJhbmssIChuYW1lLCBwKSBpbiBlbnVtZXJhdGUob3JkZXJlZCk6CiAgICAgICAgcnVubmluZyA9IG1heChydW5uaW5nLCBtaW4oMS4wLCAobGVuKG9yZGVyZWQpIC0gcmFuaykgKiBwKSkKICAgICAgICBhZGp1c3RlZFtuYW1lXSA9IHJ1bm5pbmcKICAgIHJldHVybiBhZGp1c3RlZAoKCmRlZiByZXRyaWV2ZXJfdmVyZGljdChjb250cmFzdDogZGljdFtzdHIsIEFueV0pIC0+IHN0cjoKICAgICIiIkNsYXNzaWZ5IHJlcmFua2VyLW1pbnVzLXJldHJpZXZlciBORENHQDEwIHdpdGhvdXQgaGlkaW5nIGRlZ3JhZGF0aW9uLiIiIgogICAgbG93LCBoaWdoID0gY29udHJhc3RbImNpOTUiXQogICAgaWYgbG93ID4gMDoKICAgICAgICByZXR1cm4gIkJFQVRTX1JFVFJJRVZFUiIgaWYgaGlnaCA+PSBNSU5fTUVBTklOR0ZVTF9ORENHX0RFTFRBIGVsc2UgIkdBSU5fQkVMT1dfTUVBTklOR0ZVTCIKICAgIGlmIGhpZ2ggPCAwOgogICAgICAgIHJldHVybiAiV09SU0VfVEhBTl9SRVRSSUVWRVIiCiAgICBpZiBoaWdoIDwgTUlOX01FQU5JTkdGVUxfTkRDR19ERUxUQToKICAgICAgICByZXR1cm4gIk5PX01FQU5JTkdGVUxfR0FJTiIKICAgIHJldHVybiAiSU5DT05DTFVTSVZFIgo=", "experiment.json": "ewogICJzZWVkIjogMjAyNCwKICAidG9wX20iOiAyMCwKICAidHJhaW5fcGFpcnMiOiA1MzAsCiAgImV2YWxfdXNlcnMiOiAyNjUsCiAgInZhbGlkYXRpb25fdXNlcnMiOiAyNjUsCiAgImhpc3RvcnlfaXRlbXMiOiAzLAogICJiYXRjaF9zaXplIjogNCwKICAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIjogOCwKICAibGVhcm5pbmdfcmF0ZSI6IDAuMDAwMSwKICAic2Z0X2Vwb2NocyI6IDUsCiAgImRwb19lcG9jaHMiOiA1LAogICJkcm9wX2xhc3QiOiB0cnVlLAogICJwcmVjaXNpb24iOiAiYmYxNiIsCiAgIm1vZGVsX2lkIjogIlF3ZW4vUXdlbjIuNS1WTC0zQi1JbnN0cnVjdCIsCiAgIm1vZGVsX3JldmlzaW9uIjogIjY2Mjg1NTQ2ZDJiODIxY2Y0MjFkNGY1ZWIyNTc2MzU5ZDM3NzBjZDMiLAogICJ3ZWlnaHRzIjogWzEuMCwgMC4wLCAwLjVdLAogICJub2lzZV9zaWdtYSI6IDAuMDUsCiAgImJldGEwIjogMC4xLAogICJiZXRhX2Zsb29yIjogMS4wZS02LAogICJtYXhfcGl4ZWxzIjogMjYyMTQ0LAogICJidWRnZXRfc2Vjb25kcyI6IDY2MDAsCiAgInVwc3RyZWFtIjogewogICAgImxsbTJyZWMiOiB7CiAgICAgICJ1cmwiOiAiaHR0cHM6Ly9jb2RlbG9hZC5naXRodWIuY29tL0hhcHB5UG9pbnRlci9MTE0yUmVjL3ppcC83M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIiwKICAgICAgInNoYTI1NiI6ICI2YmNlZTdiMDllYzlhZDVmYTAxMDdhMDgzYzdiODA2MGFiYmVhZDIzZTEyNjRhNDk0ODA5MDZkZTYzOGMxY2M4IiwKICAgICAgImNvbW1pdCI6ICI3M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIgogICAgfSwKICAgICJoYW5vcmVjIjogewogICAgICAidXJsIjogImh0dHBzOi8vY29kZWxvYWQuZ2l0aHViLmNvbS93YW5neXUwNjI3L0hhTm9SZWMvemlwLzU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiLAogICAgICAic2hhMjU2IjogIjQ4YzUyZTU2MjVjZDRlMjVlODA2MDBmZWQ5YmVkOGFhZDg5MDY1NTZkMjY3Yjg3ZDUyODkwYjQ5YmYzM2Q2YTYiLAogICAgICAiY29tbWl0IjogIjU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiCiAgICB9CiAgfSwKICAiYXJ0aWZhY3RfcGlucyI6IHsKICAgICJkYXRhLnR4dCI6ICI0OGM3ZWFkOThhYmZhNzNjNTA2YmY3YWYxMzM3MDNmMmVlN2U5MjIxNjdkOTIyNzYwYmIwNzQzNzY2MzlhNzJjIiwKICAgICJ0cmFpbl9kYXRhLnR4dCI6ICIxMzAwZTVkZWVjMjlkNGJlZGU2ZTMyYmZhMWE1ZWFkZTU2M2Y1M2E2OTE2ZGE5NWQ2Nzc1NGVjNDBiOTQ3NWI4IiwKICAgICJ2YWxfZGF0YS50eHQiOiAiNDg0YTk3Y2ZiY2ZmMmE3OGU5MjYxMmIxNTg2M2NlMDhjODRjMjg1YzdiZmIzMjVlYzFmMzNkMzg4NGU2ZGUxNiIsCiAgICAidGVzdF9kYXRhLnR4dCI6ICI4MDA3NGZmZWEzNzkyOGQ5MmMzNTAzOGU5ZDhiYmYzZGUwY2ZjZjAxMzA4YWU0MTczZTNjNjgwYzFiYzUwZDFmIiwKICAgICJpdGVtX3RpdGxlcy5qc29uIjogIjE3ZjUwMTgwOWM0MTU5OTA1Y2ZhNmRmYTU2MjZkM2Q1MmQwYmUzMmJjOTc0N2VlODliMDAzNjYyZjZmNWU4NGMiLAogICAgIkxMTTJSZWMtYnVkZ2V0ZWQtfGthZ2dsZXx3b3JraW5nfExMTTJSZWN8cmVwZWF0ZWRfZXZhbHVhdGVfd2l0aF9zZXEtU2VwLTA5LTIwMjZfMTEtMTItMTMtYzkzZjEyLnB0aCI6ICJmYjBkMWQzOTE2ZjEwYjViNmYwZDlmNTk3ZTdiMmFjODQzN2Y0NDVjMTU2MjllYmYzMDk5MzMzYjcyODA2YTU5IiwKICAgICJRd2VuMi0wLjVCLUxMTTJSZWMtSUVNLWJ1ZGdldGVkX3N0ZXA1MDBfdGl0bGVfaXRlbV9lbWJzLm5weSI6ICI4OWY5ZDNkZDE3ZjllYzQ5NTM3ZjkxMGUzOWM5NzYxODc4Mjk5ZjdkMjRhYmRjZjgwNWFiMGRiOTE0MzNjMGYwIiwKICAgICJnYW1lc19zYXNyZWNfc3RlcDUwMC5sb2ciOiAiYTYwMTgxYTI2MTE0MDcyZGQ1NDQyZmQ5ZmZlNGIzMzg1MDUwMTBkOGIzMjFkOWUzMWNhMTY0NmIzNDFmZjE1NCIsCiAgICAiVmlkZW9fR2FtZXNfaW1hZ2VfbWFuaWZlc3QuanNvbmwiOiAiNWQ5YzU4NmQyZDA1YzZmZjIwOGM3YTBmOTQ1YTQ0MWE0ZmM2MGVmMGQxOGVjOGFhY2MyNjlmYmY2N2E0MmQwNCIKICB9LAogICJjaGVja3BvaW50X25hbWUiOiAiTExNMlJlYy1idWRnZXRlZC18a2FnZ2xlfHdvcmtpbmd8TExNMlJlY3xyZXBlYXRlZF9ldmFsdWF0ZV93aXRoX3NlcS1TZXAtMDktMjAyNl8xMS0xMi0xMy1jOTNmMTIucHRoIiwKICAiZW1iZWRkaW5nX25hbWUiOiAiUXdlbjItMC41Qi1MTE0yUmVjLUlFTS1idWRnZXRlZF9zdGVwNTAwX3RpdGxlX2l0ZW1fZW1icy5ucHkiLAogICJwdXJwb3NlIjogImNvcnJlY3RlZCBHYW1lcyBhZGFwdGF0aW9uOyBmdWxsIHNpZ25hbCByZXF1aXJlcyBQaGFzZSAzIHF1b3RhIGdhdGUiCn0K", "audit.py": "IiIiRmFpbC1jbG9zZWQgYXVkaXRzIGZvciB0aGUgY29ycmVjdGVkIEhhTm9SZWMgR2FtZXMgc3R1ZHkuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgcmFuZG9tCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKZGVmIF9sb2FkKHBhdGg6IFBhdGgpIC0+IEFueToKICAgIHJldHVybiBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQoKCmRlZiBwcm90b2NvbF9lcnJvcnMocHJvdG9jb2w6IGRpY3Rbc3RyLCBBbnldKSAtPiBsaXN0W3N0cl06CiAgICBlcnJvcnM6IGxpc3Rbc3RyXSA9IFtdCiAgICB0YXNrID0gcHJvdG9jb2wuZ2V0KCJ0YXNrIiwge30pCiAgICByZWNpcGUgPSBwcm90b2NvbC5nZXQoInJlY2lwZSIsIHt9KQogICAgc291cmNlID0gcHJvdG9jb2wuZ2V0KCJzb3VyY2VfYXV0aG9yaXR5Iiwge30pCiAgICBpZiBub3QgdGFzay5nZXQoInByaW1hcnlfY29udHJhc3RzIik6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgicHJpbWFyeSBjb250cmFzdHMgYXJlIG1pc3NpbmciKQogICAgaWYgZmxvYXQodGFzay5nZXQoIm1lYW5pbmdmdWxfbmRjZ19kZWx0YSIsIDApKSA8PSAwOgogICAgICAgIGVycm9ycy5hcHBlbmQoIm1lYW5pbmdmdWwgTkRDRyB0aHJlc2hvbGQgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBpZiBzb3VyY2UuZ2V0KCJoYW5vcmVjIiwge30pLmdldCgiY29tbWl0IikgIT0gIjU4N2ZhY2U3NDUyNGU0NTUzYjVhN2FhMjk1ZmU5NjIwMDQ2ODIzODIiOgogICAgICAgIGVycm9ycy5hcHBlbmQoIkhhTm9SZWMgc291cmNlIGNvbW1pdCBpcyBub3QgcGlubmVkIikKICAgIGlmIHNvdXJjZS5nZXQoImxsbTJyZWMiLCB7fSkuZ2V0KCJjb21taXQiKSAhPSAiNzNiNDgxZjcxMGY2NzE2NmFiOTU4ZjQ5ODVkMjdiMjdmYjQxMDg3MSI6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgiTExNMlJlYyBzb3VyY2UgY29tbWl0IGlzIG5vdCBwaW5uZWQiKQogICAgZm9yIHN0YWdlIGluICgic2Z0IiwgImRwbyIpOgogICAgICAgIGNvbmZpZyA9IHJlY2lwZS5nZXQoc3RhZ2UsIHt9KQogICAgICAgIGZvciBmaWVsZCBpbiAoImJhdGNoX3NpemUiLCAiZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIiwgImVwb2NocyIsICJtYXhfZ3JhZF9ub3JtIik6CiAgICAgICAgICAgIGlmIGZpZWxkIG5vdCBpbiBjb25maWc6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie3N0YWdlfS57ZmllbGR9IGlzIG1pc3NpbmciKQogICAgICAgIGlmIHN0YWdlID09ICJzZnQiIGFuZCBjb25maWcuZ2V0KCJwcmVjaXNpb24iKSAhPSAiYmYxNiI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoIlNGVCBwcmVjaXNpb24gbXVzdCByZW1haW4gYmYxNiIpCiAgICBpZiByZWNpcGUuZ2V0KCJkcG8iLCB7fSkuZ2V0KCJyZWZlcmVuY2UiKSAhPSAiZnJvemVuIHBvc3QtU0ZUIGFkYXB0ZXIsIG5vIERQTyBhZGFwdGVyIjoKICAgICAgICBlcnJvcnMuYXBwZW5kKCJEUE8gcmVmZXJlbmNlIGNvbnRyYWN0IGlzIG5vdCBmcm96ZW4iKQogICAgaWYgcHJvdG9jb2wuZ2V0KCJzZWVkcyIpICE9IFsyMDI0LCAyMDI1LCAyMDI2XToKICAgICAgICBlcnJvcnMuYXBwZW5kKCJ0cmFpbmluZyBzZWVkcyBtdXN0IGJlIGV4YWN0bHkgMjAyNCwgMjAyNSwgMjAyNiIpCiAgICBpZiBzb3J0ZWQocHJvdG9jb2wuZ2V0KCJ3ZWlnaHRzIiwgW10pKSAhPSBbMC4wLCAwLjUsIDEuMF06CiAgICAgICAgZXJyb3JzLmFwcGVuZCgid2VpZ2h0cyBtdXN0IGNvbnRhaW4gMCwgMC41LCBhbmQgMSIpCiAgICBpZiAiaW52YXJpYW50cyIgbm90IGluIHByb3RvY29sIG9yIGxlbihwcm90b2NvbFsiaW52YXJpYW50cyJdKSA8IDg6CiAgICAgICAgZXJyb3JzLmFwcGVuZCgicHJvdG9jb2wgaW52YXJpYW50cyBhcmUgaW5jb21wbGV0ZSIpCiAgICBzbW9rZSA9IHByb3RvY29sLmdldCgic21va2UiLCB7fSkKICAgIGlmIG5vdCBzbW9rZS5nZXQoIm11c3Rfbm90X2JlX3VzZWRfYXNfc2lnbmFsIiwgRmFsc2UpOgogICAgICAgIGVycm9ycy5hcHBlbmQoInNtb2tlIG11c3QgYmUgZXhwbGljaXRseSBub24tc2lnbmFsIikKICAgIHJldHVybiBlcnJvcnMKCgpkZWYgYXVkaXRfcHJvdG9jb2wocGF0aDogUGF0aCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBwcm90b2NvbCA9IF9sb2FkKHBhdGgpCiAgICBlcnJvcnMgPSBwcm90b2NvbF9lcnJvcnMocHJvdG9jb2wpCiAgICByZXN1bHQgPSB7InN0YXR1cyI6ICJQQVNTIiBpZiBub3QgZXJyb3JzIGVsc2UgIkZBSUwiLCAiZXJyb3JzIjogZXJyb3JzLCAicHJvdG9jb2wiOiBzdHIocGF0aCl9CiAgICBwcmludChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBfZmluZChyb290OiBQYXRoLCBuYW1lOiBzdHIpIC0+IGxpc3RbUGF0aF06CiAgICByZXR1cm4gc29ydGVkKHBhdGggZm9yIHBhdGggaW4gcm9vdC5yZ2xvYihuYW1lKSBpZiBwYXRoLmlzX2ZpbGUoKSkKCgpkZWYgYXVkaXRfYXJ0aWZhY3RzKHJvb3Q6IFBhdGgsIHByb3RvY29sOiBkaWN0W3N0ciwgQW55XSkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXF1aXJlZCA9IFsic2Z0X2J1bmRsZS5qc29uIiwgInNmdF9sb3JhX3N0YXRlLnB0Il0KICAgIG1pc3NpbmcgPSBbbmFtZSBmb3IgbmFtZSBpbiByZXF1aXJlZCBpZiBub3QgX2ZpbmQocm9vdCwgbmFtZSldCiAgICBhcm1fcmVzdWx0cyA9IF9maW5kKHJvb3QsICJhcm1fcmVzdWx0Lmpzb24iKQogICAgZXJyb3JzID0gW2YibWlzc2luZyB7bmFtZX0iIGZvciBuYW1lIGluIG1pc3NpbmddCiAgICBleHBlY3RlZCA9IHsKICAgICAgICAoZmxvYXQod2VpZ2h0KSwgY29uZGl0aW9uKQogICAgICAgIGZvciB3ZWlnaHQgaW4gcHJvdG9jb2xbIndlaWdodHMiXQogICAgICAgIGZvciBjb25kaXRpb24gaW4gcHJvdG9jb2xbImNvbnRyb2xzIl1bImltYWdlX2NvbmRpdGlvbnMiXQogICAgfQogICAgc21va2VfbWFuaWZlc3QgPSByb290IC8gInNtb2tlX21hbmlmZXN0Lmpzb24iCiAgICBpZiBzbW9rZV9tYW5pZmVzdC5leGlzdHMoKToKICAgICAgICBzbW9rZSA9IF9sb2FkKHNtb2tlX21hbmlmZXN0KQogICAgICAgIGlmIHNtb2tlLmdldCgibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiKSBpcyBub3QgVHJ1ZToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZCgic21va2UgbWFuaWZlc3QgaXMgbm90IGV4cGxpY2l0bHkgbm9uLXNpZ25hbCIpCiAgICAgICAgaWYgc21va2UuZ2V0KCJwdXJwb3NlIikgIT0gImNvcnJlY3RuZXNzX29ubHlfbm9uX3NpZ25hbCI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoInNtb2tlIHB1cnBvc2UgaXMgbm90IGNvcnJlY3RuZXNzLW9ubHkiKQogICAgICAgIGV4cGVjdGVkID0geygxLjAsICJyZWFsIil9CiAgICAgICAgZm9yIGxhYmVsIGluICgic2Z0X2dwdV9tZW1vcnkiLCAiZHBvX2dwdV9tZW1vcnkiKToKICAgICAgICAgICAgbWVtb3J5ID0gc21va2UuZ2V0KGxhYmVsLCB7fSkKICAgICAgICAgICAgdG90YWwgPSBpbnQobWVtb3J5LmdldCgidG90YWxfYnl0ZXMiLCAwKSkKICAgICAgICAgICAgYWxsb2NhdGVkID0gaW50KG1lbW9yeS5nZXQoInBlYWtfYWxsb2NhdGVkX2J5dGVzIiwgMCkpCiAgICAgICAgICAgIHJlc2VydmVkID0gaW50KG1lbW9yeS5nZXQoInBlYWtfcmVzZXJ2ZWRfYnl0ZXMiLCAwKSkKICAgICAgICAgICAgaWYgbm90ICgwIDwgYWxsb2NhdGVkIDw9IHJlc2VydmVkIDw9IHRvdGFsKToKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJpbnZhbGlkIHtsYWJlbH0gdGVsZW1ldHJ5IikKICAgICAgICBmb3IgbGFiZWwsIHByZWZpeCBpbiAoKCJzZnRfc3RhZ2VfdGltaW5ncyIsICJzZnRfdXBkYXRlIiksICgiZHBvX3N0YWdlX3RpbWluZ3MiLCAiZHBvX3VwZGF0ZV8iKSk6CiAgICAgICAgICAgIHRpbWluZ3MgPSBzbW9rZS5nZXQobGFiZWwsIFtdKQogICAgICAgICAgICB1cGRhdGVzID0gWwogICAgICAgICAgICAgICAgcm93IGZvciByb3cgaW4gdGltaW5ncwogICAgICAgICAgICAgICAgaWYgc3RyKHJvdy5nZXQoInN0YWdlIiwgIiIpKS5zdGFydHN3aXRoKHByZWZpeCkKICAgICAgICAgICAgXQogICAgICAgICAgICBpZiBub3QgdXBkYXRlcyBvciBhbnkoZmxvYXQocm93LmdldCgic2Vjb25kcyIsIDApKSA8PSAwIGZvciByb3cgaW4gdXBkYXRlcyk6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiaW52YWxpZCB7bGFiZWx9IG9wdGltaXplciB0aW1pbmciKQogICAgb2JzZXJ2ZWQ6IHNldFt0dXBsZVtmbG9hdCwgc3RyXV0gPSBzZXQoKQogICAgZm9yIHBhdGggaW4gYXJtX3Jlc3VsdHM6CiAgICAgICAgcmVzdWx0ID0gX2xvYWQocGF0aCkKICAgICAgICBrZXkgPSAoZmxvYXQocmVzdWx0LmdldCgid2VpZ2h0IiwgZmxvYXQoIm5hbiIpKSksIHN0cihyZXN1bHQuZ2V0KCJpbWFnZV9jb25kaXRpb24iLCAiIikpKQogICAgICAgIGlmIGtleSBpbiBvYnNlcnZlZDoKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmImR1cGxpY2F0ZSBhcm06IHtrZXl9IikKICAgICAgICBvYnNlcnZlZC5hZGQoa2V5KQogICAgICAgIGlmIHJlc3VsdC5nZXQoInN0YXR1cyIpICE9ICJDT01QTEVURSI6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJpbmNvbXBsZXRlIGFybToge3BhdGh9IikKICAgICAgICBwcmVkaWN0aW9ucyA9IHJlc3VsdC5nZXQoInByZWRpY3Rpb25zIiwgW10pCiAgICAgICAgaWYgbm90IHByZWRpY3Rpb25zOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiYXJtIGhhcyBubyBwcmVkaWN0aW9uczoge3BhdGh9IikKICAgICAgICBmb3Igcm93IGluIHByZWRpY3Rpb25zOgogICAgICAgICAgICBmb3IgbWV0cmljIGluICgibmRjZ0AxMCIsICJyZWNhbGxAMTAiLCAiY2FuZGlkYXRlX3JlY2FsbEAyMCIpOgogICAgICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoZmxvYXQocm93LmdldChtZXRyaWMsIGZsb2F0KCJuYW4iKSkpKToKICAgICAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYibm9uLWZpbml0ZSB7bWV0cmljfToge3BhdGh9IikKICAgICAgICBjaGVja3BvaW50ID0gcmVzdWx0LmdldCgiY2hlY2twb2ludCIpCiAgICAgICAgY2hlY2twb2ludF9wYXRoID0gKAogICAgICAgICAgICBwYXRoLnBhcmVudCAvIGNoZWNrcG9pbnQKICAgICAgICAgICAgaWYgY2hlY2twb2ludCBhbmQgbm90IFBhdGgoY2hlY2twb2ludCkuaXNfYWJzb2x1dGUoKQogICAgICAgICAgICBlbHNlIFBhdGgoY2hlY2twb2ludCkgaWYgY2hlY2twb2ludCBlbHNlIE5vbmUKICAgICAgICApCiAgICAgICAgaWYgY2hlY2twb2ludF9wYXRoIGlzIE5vbmUgb3Igbm90IGNoZWNrcG9pbnRfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgY2hlY2twb2ludDoge3BhdGh9IikKICAgICAgICBpZiBub3QgcmVzdWx0LmdldCgicGFyZW50X3NmdF9zaGEyNTYiKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgcGFyZW50IGhhc2g6IHtwYXRofSIpCiAgICBpZiBvYnNlcnZlZCAhPSBleHBlY3RlZDoKICAgICAgICBlcnJvcnMuYXBwZW5kKGYiYXJtIG1hdHJpeCBtaXNtYXRjaDogZXhwZWN0ZWQge3NvcnRlZChleHBlY3RlZCl9LCBvYnNlcnZlZCB7c29ydGVkKG9ic2VydmVkKX0iKQogICAgb3V0cHV0ID0gewogICAgICAgICJzdGF0dXMiOiAiUEFTUyIgaWYgbm90IGVycm9ycyBlbHNlICJGQUlMIiwKICAgICAgICAiZXJyb3JzIjogZXJyb3JzLAogICAgICAgICJzZnRfYnVuZGxlX2NvdW50IjogbGVuKF9maW5kKHJvb3QsICJzZnRfYnVuZGxlLmpzb24iKSksCiAgICAgICAgImFybV9yZXN1bHRfY291bnQiOiBsZW4oYXJtX3Jlc3VsdHMpLAogICAgICAgICJleHBlY3RlZF9hcm1fY291bnQiOiBsZW4oZXhwZWN0ZWQpLAogICAgfQogICAgcHJpbnQoanNvbi5kdW1wcyhvdXRwdXQsIGluZGVudD0yKSkKICAgIHJldHVybiBvdXRwdXQKCgpkZWYgX3BhaXJlZF9tZWFuKHJvd3NfYTogbGlzdFtkaWN0W3N0ciwgQW55XV0sIHJvd3NfYjogbGlzdFtkaWN0W3N0ciwgQW55XV0sIG1ldHJpYzogc3RyKSAtPiBmbG9hdDoKICAgIGJ5X2tleSA9IHsocm93WyJ1c2VyX2lkIl0sIHJvd1sidGFyZ2V0Il0pOiByb3cgZm9yIHJvdyBpbiByb3dzX2J9CiAgICBkZWx0YXMgPSBbXQogICAgZm9yIHJvdyBpbiByb3dzX2E6CiAgICAgICAgb3RoZXIgPSBieV9rZXkuZ2V0KChyb3dbInVzZXJfaWQiXSwgcm93WyJ0YXJnZXQiXSkpCiAgICAgICAgaWYgb3RoZXIgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGFpcmVkIHByZWRpY3Rpb24ga2V5cyBkaWZmZXIiKQogICAgICAgIGRlbHRhcy5hcHBlbmQoZmxvYXQocm93W21ldHJpY10pIC0gZmxvYXQob3RoZXJbbWV0cmljXSkpCiAgICBpZiBub3QgZGVsdGFzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vIHBhaXJlZCBwcmVkaWN0aW9ucyIpCiAgICByZXR1cm4gc3VtKGRlbHRhcykgLyBsZW4oZGVsdGFzKQoKZGVmIF9wYWlyZWRfZGVsdGFzKHJvd3NfYTogbGlzdFtkaWN0W3N0ciwgQW55XV0sIHJvd3NfYjogbGlzdFtkaWN0W3N0ciwgQW55XV0sIG1ldHJpYzogc3RyKSAtPiBsaXN0W2Zsb2F0XToKICAgIGJ5X2tleSA9IHsocm93WyJ1c2VyX2lkIl0sIHJvd1sidGFyZ2V0Il0pOiByb3cgZm9yIHJvdyBpbiByb3dzX2J9CiAgICBkZWx0YXMgPSBbXQogICAgZm9yIHJvdyBpbiByb3dzX2E6CiAgICAgICAgb3RoZXIgPSBieV9rZXkuZ2V0KChyb3dbInVzZXJfaWQiXSwgcm93WyJ0YXJnZXQiXSkpCiAgICAgICAgaWYgb3RoZXIgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicGFpcmVkIHByZWRpY3Rpb24ga2V5cyBkaWZmZXIiKQogICAgICAgIGRlbHRhcy5hcHBlbmQoZmxvYXQocm93W21ldHJpY10pIC0gZmxvYXQob3RoZXJbbWV0cmljXSkpCiAgICBpZiBub3QgZGVsdGFzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vIHBhaXJlZCBwcmVkaWN0aW9ucyIpCiAgICByZXR1cm4gZGVsdGFzCgoKZGVmIF9ib290c3RyYXAoZGVsdGFzOiBsaXN0W2Zsb2F0XSwgcmVzYW1wbGVzOiBpbnQsIHNlZWQ6IGludCkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG1lYW5zID0gWwogICAgICAgIHN1bShkZWx0YXNbcm5nLnJhbmRyYW5nZShsZW4oZGVsdGFzKSldIGZvciBfIGluIGRlbHRhcykgLyBsZW4oZGVsdGFzKQogICAgICAgIGZvciBfIGluIHJhbmdlKHJlc2FtcGxlcykKICAgIF0KICAgIG1lYW5zLnNvcnQoKQogICAgbG93ID0gbWVhbnNbbWF4KDAsIGludCgwLjAyNSAqIHJlc2FtcGxlcykgLSAxKV0KICAgIGhpZ2ggPSBtZWFuc1ttaW4ocmVzYW1wbGVzIC0gMSwgaW50KDAuOTc1ICogcmVzYW1wbGVzKSldCiAgICBwID0gbWluKDEuMCwgMi4wICogbWluKAogICAgICAgIHN1bSh2YWx1ZSA8PSAwLjAgZm9yIHZhbHVlIGluIG1lYW5zKSAvIHJlc2FtcGxlcywKICAgICAgICBzdW0odmFsdWUgPj0gMC4wIGZvciB2YWx1ZSBpbiBtZWFucykgLyByZXNhbXBsZXMsCiAgICApKQogICAgcmV0dXJuIHsibWVhbiI6IHN1bShkZWx0YXMpIC8gbGVuKGRlbHRhcyksICJjaV9sb3ciOiBsb3csICJjaV9oaWdoIjogaGlnaCwgInBfdmFsdWUiOiBwfQoKCmRlZiBhbmFseXplKHJvb3Q6IFBhdGgsIGJvb3RzdHJhcF9yZXNhbXBsZXM6IGludCwgc2VlZDogaW50KSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGlmIGJvb3RzdHJhcF9yZXNhbXBsZXMgPCAxMDAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJvb3RzdHJhcF9yZXNhbXBsZXMgbXVzdCBiZSBhdCBsZWFzdCAxMDAwIikKICAgIGFybXM6IGRpY3RbdHVwbGVbZmxvYXQsIHN0cl0sIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICBmb3IgcGF0aCBpbiBfZmluZChyb290LCAiYXJtX3Jlc3VsdC5qc29uIik6CiAgICAgICAgcmVzdWx0ID0gX2xvYWQocGF0aCkKICAgICAgICBrZXkgPSAoZmxvYXQocmVzdWx0WyJ3ZWlnaHQiXSksIHN0cihyZXN1bHRbImltYWdlX2NvbmRpdGlvbiJdKSkKICAgICAgICBpZiBrZXkgaW4gYXJtczoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiZHVwbGljYXRlIGFybSB7a2V5fSIpCiAgICAgICAgYXJtc1trZXldID0gcmVzdWx0CiAgICByZXF1aXJlZCA9IFsKICAgICAgICAoMC4wLCAicmVhbCIpLCAoMC41LCAicmVhbCIpLCAoMS4wLCAicmVhbCIpLAogICAgICAgICgwLjAsICJzaHVmZmxlIiksICgwLjUsICJzaHVmZmxlIiksICgxLjAsICJzaHVmZmxlIiksCiAgICBdCiAgICBtaXNzaW5nID0gW2tleSBmb3Iga2V5IGluIHJlcXVpcmVkIGlmIGtleSBub3QgaW4gYXJtc10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYibWlzc2luZyBmaW5hbCBhcm1zOiB7bWlzc2luZ30iKQogICAgY29udHJhc3RzID0gW10KICAgIGZvciBpbmRleCwgd2VpZ2h0IGluIGVudW1lcmF0ZSgoMC4wLCAwLjUpKToKICAgICAgICByZWFsID0gYXJtc1sod2VpZ2h0LCAicmVhbCIpXVsicHJlZGljdGlvbnMiXQogICAgICAgIGJhc2VsaW5lID0gYXJtc1soMS4wLCAicmVhbCIpXVsicHJlZGljdGlvbnMiXQogICAgICAgIG5kY2cgPSBfYm9vdHN0cmFwKF9wYWlyZWRfZGVsdGFzKHJlYWwsIGJhc2VsaW5lLCAibmRjZ0AxMCIpLCBib290c3RyYXBfcmVzYW1wbGVzLCBzZWVkICsgaW5kZXgpCiAgICAgICAgcmVjYWxsID0gX2Jvb3RzdHJhcChfcGFpcmVkX2RlbHRhcyhyZWFsLCBiYXNlbGluZSwgInJlY2FsbEAxMCIpLCBib290c3RyYXBfcmVzYW1wbGVzLCBzZWVkICsgMTAwICsgaW5kZXgpCiAgICAgICAgY29udHJhc3RzLmFwcGVuZCh7IndlaWdodCI6IHdlaWdodCwgImNvbnRyYXN0IjogInctdnMtdzEiLCAibmRjZ0AxMCI6IG5kY2csICJyZWNhbGxAMTAiOiByZWNhbGx9KQogICAgcF92YWx1ZXMgPSBzb3J0ZWQoKGl0ZW1bIm5kY2dAMTAiXVsicF92YWx1ZSJdLCBpbmRleCkgZm9yIGluZGV4LCBpdGVtIGluIGVudW1lcmF0ZShjb250cmFzdHMpKQogICAgaG9sbSA9IHt9CiAgICB0b3RhbCA9IGxlbihwX3ZhbHVlcykKICAgIGZvciByYW5rLCAocF92YWx1ZSwgaW5kZXgpIGluIGVudW1lcmF0ZShwX3ZhbHVlcyk6CiAgICAgICAgaG9sbVtzdHIoY29udHJhc3RzW2luZGV4XVsid2VpZ2h0Il0pXSA9IG1pbigxLjAsICh0b3RhbCAtIHJhbmspICogcF92YWx1ZSkKICAgIG91dHB1dCA9IHsKICAgICAgICAic3RhdHVzIjogIkNPTVBMRVRFIiwKICAgICAgICAiYm9vdHN0cmFwX3Jlc2FtcGxlcyI6IGJvb3RzdHJhcF9yZXNhbXBsZXMsCiAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICJob2xtX2FkanVzdGVkX25kY2dfcCI6IGhvbG0sCiAgICAgICAgImNvbnRyYXN0cyI6IGNvbnRyYXN0cywKICAgIH0KICAgIHByaW50KGpzb24uZHVtcHMob3V0cHV0LCBpbmRlbnQ9MikpCiAgICByZXR1cm4gb3V0cHV0CgoKZGVmIG1haW4oKSAtPiBpbnQ6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBzdWJwYXJzZXJzID0gcGFyc2VyLmFkZF9zdWJwYXJzZXJzKGRlc3Q9ImNvbW1hbmQiLCByZXF1aXJlZD1UcnVlKQogICAgcHJvdG9jb2xfcGFyc2VyID0gc3VicGFyc2Vycy5hZGRfcGFyc2VyKCJwcm90b2NvbCIpCiAgICBwcm90b2NvbF9wYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb3RvY29sIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJ0aWZhY3RzX3BhcnNlciA9IHN1YnBhcnNlcnMuYWRkX3BhcnNlcigiYXJ0aWZhY3RzIikKICAgIGFydGlmYWN0c19wYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb3RvY29sIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJ0aWZhY3RzX3BhcnNlci5hZGRfYXJndW1lbnQoIi0tcm9vdCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFuYWx5emVfcGFyc2VyID0gc3VicGFyc2Vycy5hZGRfcGFyc2VyKCJhbmFseXplIikKICAgIGFuYWx5emVfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm90b2NvbCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFuYWx5emVfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYW5hbHl6ZV9wYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvb3RzdHJhcC1yZXNhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwMCkKICAgIGFuYWx5emVfcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAyNjA5MjApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgaWYgYXJncy5jb21tYW5kID09ICJwcm90b2NvbCI6CiAgICAgICAgcmVzdWx0ID0gYXVkaXRfcHJvdG9jb2woYXJncy5wcm90b2NvbCkKICAgIGVsaWYgYXJncy5jb21tYW5kID09ICJhcnRpZmFjdHMiOgogICAgICAgIHByb3RvY29sX3Jlc3VsdCA9IGF1ZGl0X3Byb3RvY29sKGFyZ3MucHJvdG9jb2wpCiAgICAgICAgaWYgcHJvdG9jb2xfcmVzdWx0WyJzdGF0dXMiXSAhPSAiUEFTUyI6CiAgICAgICAgICAgIHJldHVybiAxCiAgICAgICAgcmVzdWx0ID0gYXVkaXRfYXJ0aWZhY3RzKGFyZ3Mucm9vdCwgX2xvYWQoYXJncy5wcm90b2NvbCkpCiAgICBlbHNlOgogICAgICAgIHByb3RvY29sX3Jlc3VsdCA9IGF1ZGl0X3Byb3RvY29sKGFyZ3MucHJvdG9jb2wpCiAgICAgICAgaWYgcHJvdG9jb2xfcmVzdWx0WyJzdGF0dXMiXSAhPSAiUEFTUyI6CiAgICAgICAgICAgIHJldHVybiAxCiAgICAgICAgcmVzdWx0ID0gYW5hbHl6ZShhcmdzLnJvb3QsIGFyZ3MuYm9vdHN0cmFwX3Jlc2FtcGxlcywgYXJncy5zZWVkKQogICAgcmV0dXJuIDAgaWYgcmVzdWx0WyJzdGF0dXMiXSBpbiB7IlBBU1MiLCAiQ09NUExFVEUifSBlbHNlIDEKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==", "run-protocol.json": "ewogICJwcm90b2NvbF92ZXJzaW9uIjogMiwKICAicGxhbiI6ICIyNjA5MjAtMTI0NS1oYW5vcmVjLWZhaXRoZnVsLXNpZ25hbC1yZWNoZWNrIiwKICAidGFzayI6IHsKICAgICJkb21haW4iOiAiR2FtZXMgYWRhcHRhdGlvbiBvZiBIYU5vUmVjIGhpdC0xIGJpbmFyeSBwcmVmZXJlbmNlIiwKICAgICJuYXRpdmVfc2NvcmVfY2xhaW0iOiBmYWxzZSwKICAgICJwcmltYXJ5X21ldHJpY3MiOiBbCiAgICAgICJORENHQDEwIiwKICAgICAgIlJlY2FsbEAxMCIKICAgIF0sCiAgICAicHJpbWFyeV9jb250cmFzdHMiOiBbCiAgICAgICJ3PTAuMCBtaW51cyB3PTEuMCIsCiAgICAgICJ3PTAuNSBtaW51cyB3PTEuMCIKICAgIF0sCiAgICAibWVhbmluZ2Z1bF9uZGNnX2RlbHRhIjogMC4wMDIsCiAgICAiY29uZmlybWF0aW9uX3NwbGl0IjogInRlc3QiLAogICAgImRldmVsb3BtZW50X3NwbGl0IjogInZhbGlkYXRpb24iLAogICAgImNhbmRpZGF0ZV9zb3VyY2UiOiAiZnJvemVuIFNBU1JlYyB0b3AtTSBjYW5kaWRhdGVzOyB0YXJnZXQgbmV2ZXIgaW5zZXJ0ZWQiLAogICAgInNjb3JpbmciOiAic2VxdWVuY2Utc3VtbWVkIFllcy9ObyByZXNwb25zZSBsb2ctcHJvYmFiaWxpdHkgbWFyZ2luIgogIH0sCiAgInNvdXJjZV9hdXRob3JpdHkiOiB7CiAgICAiaGFub3JlYyI6IHsKICAgICAgInJlcG9zaXRvcnkiOiAiaHR0cHM6Ly9naXRodWIuY29tL3dhbmd5dTA2MjcvSGFOb1JlYyIsCiAgICAgICJjb21taXQiOiAiNTg3ZmFjZTc0NTI0ZTQ1NTNiNWE3YWEyOTVmZTk2MjAwNDY4MjM4MiIsCiAgICAgICJhcmNoaXZlX3NoYTI1NiI6ICI0OGM1MmU1NjI1Y2Q0ZTI1ZTgwNjAwZmVkOWJlZDhhYWQ4OTA2NTU2ZDI2N2I4N2Q1Mjg5MGI0OWJmMzNkNmE2IiwKICAgICAgInBhdGhzIjogWwogICAgICAgICJjb25maWdzL2hhbm9yZWMvbWljcm9sZW5zX2hpdDEueWFtbCIsCiAgICAgICAgImhhbm9yZWMvdHJhaW5lci9kcG8ucHkiLAogICAgICAgICJoYW5vcmVjL2hhcnMvbWF0aC5weSIsCiAgICAgICAgImhhbm9yZWMvaGFycy9oYXJkbmVzcy5weSIsCiAgICAgICAgImhhbm9yZWMvbm9kby9ob29rcy5weSIKICAgICAgXQogICAgfSwKICAgICJsbGFtYWZhY3RvcnkiOiB7CiAgICAgICJyZXF1aXJlbWVudCI6ICJsbGFtYWZhY3Rvcnk9PTAuOS4zLmRldjAiLAogICAgICAic291cmNlX3JldmlzaW9uIjogbnVsbCwKICAgICAgIm5vdGUiOiAiVGhlIHBpbm5lZCBIYU5vUmVjIHJlcXVpcmVtZW50IGV4cG9zZXMgYSBwYWNrYWdlIHZlcnNpb24gYnV0IG5vIHNvdXJjZSBjb21taXQuIFRoZSBjb3JyZWN0ZWQgR2FtZXMgcnVubmVyIHBvcnRzIGFuZCB0ZXN0cyB0aGUgb2JzZXJ2YWJsZSBIYU5vUmVjIHRyYWluZXIgY29udHJhY3Q7IGl0IGRvZXMgbm90IGNsYWltIGJ5dGUtaWRlbnRpY2FsIExMYU1BLUZhY3RvcnkgaW50ZXJuYWxzLiIKICAgIH0sCiAgICAibGxtMnJlYyI6IHsKICAgICAgInJlcG9zaXRvcnkiOiAiaHR0cHM6Ly9naXRodWIuY29tL0hhcHB5UG9pbnRlci9MTE0yUmVjIiwKICAgICAgImNvbW1pdCI6ICI3M2I0ODFmNzEwZjY3MTY2YWI5NThmNDk4NWQyN2IyN2ZiNDEwODcxIiwKICAgICAgImFyY2hpdmVfc2hhMjU2IjogIjZiY2VlN2IwOWVjOWFkNWZhMDEwN2EwODNjN2I4MDYwYWJiZWFkMjNlMTI2NGE0OTQ4MDkwNmRlNjM4YzFjYzgiCiAgICB9CiAgfSwKICAicmVjaXBlIjogewogICAgIm1vZGVsIjogIlF3ZW4vUXdlbjIuNS1WTC0zQi1JbnN0cnVjdCIsCiAgICAibW9kZWxfcmV2aXNpb24iOiAiNjYyODU1NDZkMmI4MjFjZjQyMWQ0ZjVlYjI1NzYzNTlkMzc3MGNkMyIsCiAgICAic2Z0IjogewogICAgICAiYWRhcHRlciI6ICJ0YXNrLXNwZWNpZmljIExvUkEgYWRhcHRlciIsCiAgICAgICJiYXRjaF9zaXplIjogMiwKICAgICAgImdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyI6IDgsCiAgICAgICJlcG9jaHMiOiA1LjAsCiAgICAgICJzY2hlZHVsZXIiOiAiY29zaW5lIiwKICAgICAgIndhcm11cF9zdGVwcyI6IDAsCiAgICAgICJtYXhfZ3JhZF9ub3JtIjogMS4wLAogICAgICAicHJlY2lzaW9uIjogImJmMTYiLAogICAgICAibG9yYV9yYW5rIjogOCwKICAgICAgImxvcmFfYWxwaGEiOiAzMiwKICAgICAgImxvcmFfZHJvcG91dCI6IDAuMDUsCiAgICAgICJ0YXJnZXRfbW9kdWxlcyI6ICJsYW5ndWFnZS1tb2RlbCBsaW5lYXIgbGF5ZXJzIChxLGssdixvLGdhdGUsdXAsZG93bik7IHZpc2lvbiB0b3dlciBmcm96ZW4iLAogICAgICAidGFzayI6ICI1MCUgWWVzIC8gNTAlIE5vIgogICAgfSwKICAgICJkcG8iOiB7CiAgICAgICJmcmVzaF9hZGFwdGVyIjogdHJ1ZSwKICAgICAgInJlZmVyZW5jZSI6ICJmcm96ZW4gcG9zdC1TRlQgYWRhcHRlciwgbm8gRFBPIGFkYXB0ZXIiLAogICAgICAiYmF0Y2hfc2l6ZSI6IDQsCiAgICAgICJncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMiOiA4LAogICAgICAiZXBvY2hzIjogNS4wLAogICAgICAic2NoZWR1bGVyIjogImNvc2luZSIsCiAgICAgICJ3YXJtdXBfc3RlcHMiOiAwLAogICAgICAibWF4X2dyYWRfbm9ybSI6IDEuMCwKICAgICAgImJldGEwIjogMC4xLAogICAgICAibG9zcyI6ICJzaWdtb2lkIiwKICAgICAgIm5vaXNlX3NpZ21hIjogMC4wNSwKICAgICAgImJldGFfZmxvb3IiOiAxZS0wNiwKICAgICAgImRyb3BfbGFzdCI6IHRydWUsCiAgICAgICJkcm9wb3V0IjogImRpc2FibGVkIiwKICAgICAgInRhcmdldF9tb2R1bGVzIjogImxhbmd1YWdlLW1vZGVsIGxpbmVhciBsYXllcnMiCiAgICB9CiAgfSwKICAic2VlZHMiOiBbCiAgICAyMDI0LAogICAgMjAyNSwKICAgIDIwMjYKICBdLAogICJ3ZWlnaHRzIjogWwogICAgMS4wLAogICAgMC4wLAogICAgMC41CiAgXSwKICAiY29udHJvbHMiOiB7CiAgICAiaW1hZ2VfY29uZGl0aW9ucyI6IFsKICAgICAgInJlYWwiLAogICAgICAic2h1ZmZsZSIKICAgIF0sCiAgICAic2h1ZmZsZSI6ICJmaXhlZC1wb2ludC1mcmVlIGZyZXF1ZW5jeS1zdHJhdGlmaWVkIGl0ZW0gYmlqZWN0aW9uIGJ1aWx0IGZyb20gdHJhaW5pbmcgcm93cyBvbmx5IiwKICAgICJjb25zdGFudF9oYXJkbmVzcyI6ICJ2YWxpZGF0aW9uLW9ubHkgbWVhbi1tYXRjaGVkIGRpYWdub3N0aWM7IG5vdCBhIHByaW1hcnkgdGVzdCBhcm0iCiAgfSwKICAic21va2UiOiB7CiAgICAicHVycG9zZSI6ICJjb3JyZWN0bmVzcyBhbmQgbWVhc3VyZWQgcmVzb3VyY2Ugc21va2Ugb25seSIsCiAgICAidHJhaW5fcGFpcnMiOiA4LAogICAgInZhbGlkYXRpb25fdXNlcnMiOiA0LAogICAgInRlc3RfdXNlcnMiOiA0LAogICAgImVwb2NocyI6IDEsCiAgICAibWF4X29wdGltaXplcl91cGRhdGVzIjogMiwKICAgICJtdXN0X25vdF9iZV91c2VkX2FzX3NpZ25hbCI6IHRydWUKICB9LAogICJpbnZhcmlhbnRzIjogWwogICAgInRyYWluaW5nIHBhaXIgYW5kIGhhcmRuZXNzIGhhc2hlcyBkbyBub3QgY2hhbmdlIHdoZW4gaGVsZC1vdXQgdGFyZ2V0cy9jYW5kaWRhdGVzL2ltYWdlcyBjaGFuZ2UiLAogICAgIm5lZ2F0aXZlIGlzIG5vdCB0YXJnZXQsIG9ic2VydmVkIGhpc3RvcnksIG9yIHRyYWluLWtub3duIGZ1dHVyZSBpdGVtIiwKICAgICJzaHVmZmxlIGlzIGEgYmlqZWN0aW9uIGFuZCBoYXMgbm8gZml4ZWQgcG9pbnRzIiwKICAgICJnbG9iYWwgcmVzcG9uc2l2ZW5lc3MgYmF0Y2ggaGFzIGF0IGxlYXN0IHRocmVlIGV4YW1wbGVzIiwKICAgICJvbmx5IHRoZSBmcmVzaCBEUE8gYWRhcHRlciBjaGFuZ2VzIGR1cmluZyBEUE8iLAogICAgIlNGVC1vbmx5IHJlZmVyZW5jZSBpcyBpbnZhcmlhbnQgdG8gRFBPIHVwZGF0ZXMgYW5kIE5vRE8gbm9pc2UiLAogICAgInBhcnRpYWwgb3IgbWlzc2luZyBzdGFnZXMgY2Fubm90IGJlIGxhYmVsbGVkIENPTVBMRVRFIiwKICAgICJzb3VyY2UvY29uZmlnL2RhdGEvcGFyZW50IGhhc2hlcyBhcmUgcGVyc2lzdGVkIGFuZCBjaGVja2VkIgogIF0sCiAgInN0YXRpc3RpY3MiOiB7CiAgICAiYm9vdHN0cmFwX3Jlc2FtcGxlcyI6IDIwMDAwLAogICAgImJvb3RzdHJhcF9zZWVkIjogMjAyNjEwMDEsCiAgICAiaG9sbV9wcmltYXJ5X2NvbnRyYXN0cyI6IDIsCiAgICAidHJhaW5pbmdfc2VlZF91bmNlcnRhaW50eSI6ICJyZXBvcnQgaW5kaXZpZHVhbCBzZWVkcyBhbmQgc3ByZWFkOyBib290c3RyYXAgaXMgY29uZGl0aW9uYWwgb24gZml0dGVkIHNlZWQgbW9kZWxzIiwKICAgICJwX3ZhbHVlIjogIk1vbnRlLUNhcmxvIChjb3VudCsxKS8obisxKSIKICB9LAogICJyZXBhaXJzX3YyIjogWwogICAgIlNGVCB0YXNrIGlzIHRoZSBIYU5vUmVjIGhpdD0xIGJpbmFyeSB0YXNrOiB0cnVlIG5leHQgaXRlbSAtPiBZZXMsIHNlZWRlZCByYW5kb20gdHJhaW4tb25seSB1bnNlZW4gaXRlbSAtPiBObyAoNTAvNTApOyBoYXJkIENGIG5lZ2F0aXZlcyBhcmUgRFBPLW9ubHkiLAogICAgIkRQTyBwb2xpY3kgcnVucyB3aXRoIGFsbCBkcm9wb3V0IG1vZHVsZXMgYXQgcD0wIChMTGFNQS1GYWN0b3J5IGRpc2FibGVfZHJvcG91dF9pbl9tb2RlbCk7IHJlZmVyZW5jZSBpcyB0aGUgU0ZUIGFkYXB0ZXIgb25seSIsCiAgICAiTG9SQSB0YXJnZXRzIGxhbmd1YWdlLW1vZGVsIGxpbmVhciBsYXllcnMgb25seTsgdmlzaW9uIHRvd2VyIGFuZCBtZXJnZXIgZnJvemVuIChMTGFNQS1GYWN0b3J5IGZyZWV6ZV92aXNpb25fdG93ZXI9VHJ1ZSkiLAogICAgImNhbmRpZGF0ZSBzY29yZSB0aWVzIGtlZXAgdGhlIGZyb3plbiBTQVNSZWMgb3JkZXIgaW5zdGVhZCBvZiBpdGVtLWlkIG9yZGVyIiwKICAgICJTRlQgbWljcm8tYmF0Y2ggMiB4IGFjY3VtdWxhdGlvbiA4IGZvbGxvd3MgdHJhaW4vbWxsbV9zZnQuc2giLAogICAgIm5hdGl2ZSBiZjE2IGlzIHJlcXVpcmVkIGJ5IHRoZSBvcmlnaW5hbCBwcm90b2NvbDsgS2FnZ2xlIHNlcnZlcyBvbmx5IFQ0L1AxMDAgKGJmMTYgZW11bGF0ZWQgb24gVDQpIHNvIHRoaXMgaXMgcmVjb3JkZWQgYXMgYW4gYWRhcHRhdGlvbiwgbm90IGVuZm9yY2VkIiwKICAgICJTRlQgdXNlcyBib3RoIGEgWWVzIGFuZCBhIE5vIHNhbXBsZSBwZXIgdHJhaW5pbmcgcGFpciAoMnggZXhhbXBsZXMvZXBvY2ggdnMgdXBzdHJlYW0ncyBvbmUgcmFuZG9tbHktbGFiZWxsZWQgc2FtcGxlIHBlciByb3cpOyBlcG9jaHMgYW5kIG1heF9waXhlbHMgYXJlIGRlcml2ZWQgZnJvbSBtZWFzdXJlZCBUNCBzcGVlZC9WUkFNIGFuZCByZWNvcmRlZCBhcyBhZGFwdGF0aW9ucyIKICBdCn0K", "test_hanorec_fidelity.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwppbXBvcnQgdW5pdHRlc3QKZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCmltcG9ydCB0cmFpbiBhcyB0cmFpbl9tb2R1bGUgICMgbm9xYTogRTQwMgpmcm9tIHRyYWluIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIF9hY2N1bXVsYXRpb25fZ3JvdXBfc2l6ZSwKICAgIF9jdWRhX3BlYWtfbWVtb3J5LAogICAgX2J1aWxkX2Rwb19leGFtcGxlcywKICAgIF9jaHVua2VkLAogICAgX2Zpbml0ZSwKICAgIF9qc29uX2RlZmF1bHQsCiAgICBfcmVzZXRfY3VkYV9wZWFrX21lbW9yeSwKICAgIF9zY29yZV9yb3dzLAogICAgX3Jlc3RvcmVfYWN0aXZlX2FkYXB0ZXJzLAogICAgX3NldF9hY3RpdmVfYWRhcHRlcnMsCiAgICBfc2V0X29ubHlfdHJhaW5hYmxlX2FkYXB0ZXIsCiAgICBMT1JBX1RBUkdFVF9SRUdFWCwKICAgIF9idWlsZF9zZnRfZXhhbXBsZXMsCiAgICBfZGlzYWJsZV9kcm9wb3V0LAogICAgaGFyc19ub3JtYWxpemVfaGFyZG5lc3MsCiAgICByZXNwb25zaXZlbmVzcywKKQoKY2xhc3MgX0FkYXB0ZXJCYXNlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsKToKICAgICAgICBzZWxmLm1vZGVsID0gbW9kZWwKICAgICAgICBzZWxmLmNhbGxzID0gW10KCiAgICBkZWYgc2V0X2FkYXB0ZXIoc2VsZiwgYWRhcHRlcnMpOgogICAgICAgIG5hbWVzID0gW2FkYXB0ZXJzXSBpZiBpc2luc3RhbmNlKGFkYXB0ZXJzLCBzdHIpIGVsc2UgbGlzdChhZGFwdGVycykKICAgICAgICBzZWxmLmNhbGxzLmFwcGVuZChuYW1lcykKICAgICAgICBmb3IgbmFtZSwgcGFyYW1ldGVyIGluIHNlbGYubW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgICAgICBwYXJhbWV0ZXIucmVxdWlyZXNfZ3JhZCA9IGFueShmIi57YWRhcHRlcn0uIiBpbiBuYW1lIGZvciBhZGFwdGVyIGluIG5hbWVzKQoKY2xhc3MgX1BhcmFtZXRlcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQoKCmNsYXNzIF9BZGFwdGVyTW9kZWw6CiAgICBkZWYgX19pbml0X18oc2VsZik6CiAgICAgICAgc2VsZi5jYWxscyA9IFtdCiAgICAgICAgc2VsZi5hY3RpdmVfYWRhcHRlciA9IE5vbmUKICAgICAgICBzZWxmLnBlZnRfY29uZmlnID0geyJzZnQiOiBvYmplY3QoKSwgImRwbyI6IG9iamVjdCgpfQogICAgICAgIHNlbGYucGFyYW1ldGVycyA9IHsKICAgICAgICAgICAgImxheWVyLnNmdC5sb3JhX0EiOiBfUGFyYW1ldGVyKCksCiAgICAgICAgICAgICJsYXllci5kcG8ubG9yYV9BIjogX1BhcmFtZXRlcigpLAogICAgICAgIH0KICAgICAgICBzZWxmLmJhc2VfbW9kZWwgPSBfQWRhcHRlckJhc2Uoc2VsZikKCiAgICBkZWYgc2V0X2FkYXB0ZXIoc2VsZiwgYWRhcHRlcik6CiAgICAgICAgc2VsZi5jYWxscy5hcHBlbmQoYWRhcHRlcikKCiAgICBkZWYgbmFtZWRfcGFyYW1ldGVycyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5wYXJhbWV0ZXJzLml0ZW1zKCkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBhY3RpdmVfcGVmdF9jb25maWcoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYucGVmdF9jb25maWdbc2VsZi5hY3RpdmVfYWRhcHRlcl0KCgpjbGFzcyBfQ3VkYVByb3BlcnRpZXM6CiAgICB0b3RhbF9tZW1vcnkgPSAxXzAwMAoKCmNsYXNzIF9GYWtlQ3VkYToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnJlc2V0ID0gRmFsc2UKCiAgICBkZWYgaXNfYXZhaWxhYmxlKHNlbGYpOgogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIHN5bmNocm9uaXplKHNlbGYpOgogICAgICAgIHJldHVybiBOb25lCgogICAgZGVmIHJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKHNlbGYpOgogICAgICAgIHNlbGYucmVzZXQgPSBUcnVlCgogICAgZGVmIGN1cnJlbnRfZGV2aWNlKHNlbGYpOgogICAgICAgIHJldHVybiAwCgogICAgZGVmIGdldF9kZXZpY2VfcHJvcGVydGllcyhzZWxmLCBfZGV2aWNlKToKICAgICAgICByZXR1cm4gX0N1ZGFQcm9wZXJ0aWVzKCkKCiAgICBkZWYgZ2V0X2RldmljZV9uYW1lKHNlbGYsIF9kZXZpY2UpOgogICAgICAgIHJldHVybiAiZmFrZS1ncHUiCgogICAgZGVmIG1heF9tZW1vcnlfYWxsb2NhdGVkKHNlbGYsIF9kZXZpY2UpOgogICAgICAgIHJldHVybiA2MDAKCiAgICBkZWYgbWF4X21lbW9yeV9yZXNlcnZlZChzZWxmLCBfZGV2aWNlKToKICAgICAgICByZXR1cm4gNzUwCgoKY2xhc3MgX0Zha2VUb3JjaDoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLmN1ZGEgPSBfRmFrZUN1ZGEoKQoKCgpjbGFzcyBfTm9HcmFkOgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKToKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGRlZiBfX2V4aXRfXyhzZWxmLCAqX2FyZ3MpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmNsYXNzIF9TY29yZVRvcmNoOgogICAgZGVmIG5vX2dyYWQoc2VsZik6CiAgICAgICAgcmV0dXJuIF9Ob0dyYWQoKQoKCmNsYXNzIF9TY29yZU1vZGVsOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYudHJhaW5pbmcgPSBUcnVlCgogICAgZGVmIGV2YWwoc2VsZik6CiAgICAgICAgc2VsZi50cmFpbmluZyA9IEZhbHNlCgogICAgZGVmIHRyYWluKHNlbGYpOgogICAgICAgIHNlbGYudHJhaW5pbmcgPSBUcnVlCgoKY2xhc3MgX1Njb3JlVmFsdWU6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdmFsdWUpOgogICAgICAgIHNlbGYudmFsdWUgPSBmbG9hdCh2YWx1ZSkKCiAgICBkZWYgX19zdWJfXyhzZWxmLCBvdGhlcik6CiAgICAgICAgcmV0dXJuIF9TY29yZVZhbHVlKHNlbGYudmFsdWUgLSBvdGhlci52YWx1ZSkKCiAgICBkZWYgY3B1KHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLnZhbHVlCmNsYXNzIF9TY2FsYXI6CiAgICBkZWYgaXRlbShzZWxmKToKICAgICAgICByZXR1cm4gMS41CgoKY2xhc3MgSGFOb1JlY0ZpZGVsaXR5VGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfdXBzdHJlYW1fcmVzcG9uc2l2ZW5lc3NfaGFzX2dsb2JhbF90cmltbWVkX2JhdGNoX3NlbWFudGljcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHZhbHVlID0gcmVzcG9uc2l2ZW5lc3MoWzEuMCwgMi4wLCAzLjAsIDEwMC4wXSkKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIodmFsdWUsIDAuMCkKICAgICAgICBzZWxmLmFzc2VydExlc3ModmFsdWUsIDIuMCkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICByZXNwb25zaXZlbmVzcyhbMS4wLCBmbG9hdCgibmFuIiksIDMuMF0pCgogICAgZGVmIHRlc3RfZHBvX2V4YW1wbGVzX3ByZXNlcnZlX2JpbmFyeV9wYWlyX29yZGVyKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZXhhbXBsZXMgPSBfYnVpbGRfZHBvX2V4YW1wbGVzKAogICAgICAgICAgICBbeyJoaXN0b3J5IjogWzEsIDIsIDNdLCAicG9zaXRpdmUiOiA0LCAibmVnYXRpdmUiOiA1fV0sCiAgICAgICAgICAgIHllc190b2tlbl9pZD0xMCwKICAgICAgICAgICAgbm9fdG9rZW5faWQ9MTEsCiAgICAgICAgICAgIHNodWZmbGVfbWFwPU5vbmUsCiAgICAgICAgKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoCiAgICAgICAgICAgIFsocm93WyJjYW5kaWRhdGUiXSwgcm93WyJjaG9zZW5fdG9rZW4iXSwgcm93WyJyZWplY3RlZF90b2tlbiJdKSBmb3Igcm93IGluIGV4YW1wbGVzXSwKICAgICAgICAgICAgWyg0LCAxMCwgMTEpLCAoNSwgMTEsIDEwKV0sCiAgICAgICAgKQoKICAgIGRlZiB0ZXN0X2JhdGNoX2JvdW5kYXJpZXNfYW5kX2hhcmRuZXNzX2FyZV9mYWlsX2Nsb3NlZChzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoW2xlbihiYXRjaCkgZm9yIGJhdGNoIGluIF9jaHVua2VkKGxpc3QocmFuZ2UoMTApKSwgNCwgVHJ1ZSldLCBbNCwgNF0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChbbGVuKGJhdGNoKSBmb3IgYmF0Y2ggaW4gX2NodW5rZWQobGlzdChyYW5nZSgxMCkpLCA0LCBGYWxzZSldLCBbNCwgNCwgMl0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oaGFyc19ub3JtYWxpemVfaGFyZG5lc3MoWzAuMCwgMS4wLCAyLjBdKSksIDMpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgX2NodW5rZWQoWzEsIDJdLCAwLCBUcnVlKQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIF9maW5pdGUoW10sICJ2YWx1ZXMiKQoKICAgIGRlZiB0ZXN0X211bHRpX2FkYXB0ZXJfYWN0aXZhdGlvbl9mcmVlemVzX3NmdF9wYXJhbWV0ZXJzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBfQWRhcHRlck1vZGVsKCkKICAgICAgICBfc2V0X2FjdGl2ZV9hZGFwdGVycyhtb2RlbCwgWyJzZnQiLCAiZHBvIl0pCiAgICAgICAgX3NldF9vbmx5X3RyYWluYWJsZV9hZGFwdGVyKG1vZGVsLCAiZHBvIikKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKG1vZGVsLnBhcmFtZXRlcnNbImxheWVyLnNmdC5sb3JhX0EiXS5yZXF1aXJlc19ncmFkKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShtb2RlbC5wYXJhbWV0ZXJzWyJsYXllci5kcG8ubG9yYV9BIl0ucmVxdWlyZXNfZ3JhZCkKICAgICAgICBfcmVzdG9yZV9hY3RpdmVfYWRhcHRlcnMobW9kZWwsIFsic2Z0IiwgImRwbyJdKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UobW9kZWwucGFyYW1ldGVyc1sibGF5ZXIuc2Z0LmxvcmFfQSJdLnJlcXVpcmVzX2dyYWQpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKG1vZGVsLnBhcmFtZXRlcnNbImxheWVyLmRwby5sb3JhX0EiXS5yZXF1aXJlc19ncmFkKQoKICAgIGRlZiB0ZXN0X211bHRpX2FkYXB0ZXJfYWN0aXZhdGlvbl91c2VzX3BlZnRfYmFzZV9tb2RlbChzZWxmKSAtPiBOb25lOgogICAgICAgIG1vZGVsID0gX0FkYXB0ZXJNb2RlbCgpCiAgICAgICAgX3NldF9hY3RpdmVfYWRhcHRlcnMobW9kZWwsIFsic2Z0IiwgImRwbyJdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobW9kZWwuYmFzZV9tb2RlbC5jYWxscywgW1sic2Z0IiwgImRwbyJdXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG1vZGVsLmFjdGl2ZV9hZGFwdGVyLCAiZHBvIikKICAgICAgICBzZWxmLmFzc2VydElzKG1vZGVsLmFjdGl2ZV9wZWZ0X2NvbmZpZywgbW9kZWwucGVmdF9jb25maWdbImRwbyJdKQogICAgICAgIF9zZXRfYWN0aXZlX2FkYXB0ZXJzKG1vZGVsLCAic2Z0IikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG1vZGVsLmNhbGxzLCBbInNmdCJdKQogICAgZGVmIHRlc3RfcGFydGlhbF9hY2N1bXVsYXRpb25fdXNlc19vbmVfZ3JvdXBfZGl2aXNvcihzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoCiAgICAgICAgICAgIFtfYWNjdW11bGF0aW9uX2dyb3VwX3NpemUocG9zaXRpb24sIDEwLCA0KSBmb3IgcG9zaXRpb24gaW4gcmFuZ2UoMTApXSwKICAgICAgICAgICAgWzQsIDQsIDQsIDQsIDQsIDQsIDQsIDQsIDIsIDJdLAogICAgICAgICkKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBfYWNjdW11bGF0aW9uX2dyb3VwX3NpemUoMCwgMCwgNCkKCgogICAgZGVmIHRlc3RfY3VkYV9wZWFrX21lbW9yeV9yZXBvcnRzX2NhcGFjaXR5X2ZyYWN0aW9ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIHRvcmNoX21vZHVsZSA9IF9GYWtlVG9yY2goKQogICAgICAgIF9yZXNldF9jdWRhX3BlYWtfbWVtb3J5KHRvcmNoX21vZHVsZSkKICAgICAgICBzZWxmLmFzc2VydFRydWUodG9yY2hfbW9kdWxlLmN1ZGEucmVzZXQpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCgKICAgICAgICAgICAgX2N1ZGFfcGVha19tZW1vcnkodG9yY2hfbW9kdWxlKSwKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImRldmljZSI6ICJmYWtlLWdwdSIsCiAgICAgICAgICAgICAgICAidG90YWxfYnl0ZXMiOiAxXzAwMCwKICAgICAgICAgICAgICAgICJwZWFrX2FsbG9jYXRlZF9ieXRlcyI6IDYwMCwKICAgICAgICAgICAgICAgICJwZWFrX3Jlc2VydmVkX2J5dGVzIjogNzUwLAogICAgICAgICAgICAgICAgInBlYWtfYWxsb2NhdGVkX2ZyYWN0aW9uIjogMC42LAogICAgICAgICAgICAgICAgInBlYWtfcmVzZXJ2ZWRfZnJhY3Rpb24iOiAwLjc1LAogICAgICAgICAgICB9LAogICAgICAgICkKCiAgICBkZWYgdGVzdF9jYW5kaWRhdGVfc2NvcmluZ19iYXRjaGVzX3dpdGhvdXRfY2hhbmdpbmdfcmFua19jb250cmFjdChzZWxmKSAtPiBOb25lOgogICAgICAgIG1vZGVsID0gX1Njb3JlTW9kZWwoKQogICAgICAgIGJhdGNoX3NpemVzID0gW10KCiAgICAgICAgZGVmIGZha2VfaW5wdXRzKF9wcm9jZXNzb3IsIF9tb2RlbCwgX2NhdGFsb2csIGV4YW1wbGVzLCBfcGl4ZWxzLCBfc2h1ZmZsZSk6CiAgICAgICAgICAgIGJhdGNoX3NpemVzLmFwcGVuZChsZW4oZXhhbXBsZXMpKQogICAgICAgICAgICByZXR1cm4gZXhhbXBsZXMKCiAgICAgICAgZGVmIGZha2VfbG9ncHJvYnMoX21vZGVsLCBleGFtcGxlcywgX2Z1bmN0aW9uYWwsIF95ZXMsIF9ubyk6CiAgICAgICAgICAgIHJldHVybiBbKF9TY29yZVZhbHVlKGV4YW1wbGVbImNhbmRpZGF0ZSJdKSwgX1Njb3JlVmFsdWUoMC4wKSkgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXNdCgogICAgICAgIHdpdGggKAogICAgICAgICAgICBwYXRjaC5vYmplY3QodHJhaW5fbW9kdWxlLCAiX2J1aWxkX2JhdGNoX2lucHV0cyIsIHNpZGVfZWZmZWN0PWZha2VfaW5wdXRzKSwKICAgICAgICAgICAgcGF0Y2gub2JqZWN0KHRyYWluX21vZHVsZSwgIl9iYXRjaF9hbnN3ZXJfbG9ncHJvYnMiLCBzaWRlX2VmZmVjdD1mYWtlX2xvZ3Byb2JzKSwKICAgICAgICApOgogICAgICAgICAgICByZXN1bHQgPSBfc2NvcmVfcm93cygKICAgICAgICAgICAgICAgIFt7CiAgICAgICAgICAgICAgICAgICAgInVzZXJfaWQiOiAidSIsCiAgICAgICAgICAgICAgICAgICAgImhpc3RvcnkiOiBbOSwgOCwgN10sCiAgICAgICAgICAgICAgICAgICAgInRhcmdldCI6IDIsCiAgICAgICAgICAgICAgICAgICAgImNhbmRpZGF0ZXMiOiBbMSwgMywgMl0sCiAgICAgICAgICAgICAgICB9XSwKICAgICAgICAgICAgICAgIG9iamVjdCgpLCBtb2RlbCwge30sIDgxOTIsIDEsIDAsIF9TY29yZVRvcmNoKCksIG9iamVjdCgpLCBOb25lLAogICAgICAgICAgICAgICAgZXZhbF9iYXRjaF9zaXplPTIsCiAgICAgICAgICAgICkKCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChiYXRjaF9zaXplcywgWzIsIDFdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WzBdWyJyYW5rZWRfY2FuZGlkYXRlcyJdLCBbMywgMiwgMV0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbMF1bInJhbmsiXSwgMikKICAgICAgICBzZWxmLmFzc2VydFRydWUobW9kZWwudHJhaW5pbmcpCgogICAgZGVmIHRlc3RfZXF1YWxfc2NvcmVzX2tlZXBfcmV0cmlldmVyX29yZGVyX25vdF9pdGVtX2lkX29yZGVyKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBfU2NvcmVNb2RlbCgpCiAgICAgICAgd2l0aCAoCiAgICAgICAgICAgIHBhdGNoLm9iamVjdCh0cmFpbl9tb2R1bGUsICJfYnVpbGRfYmF0Y2hfaW5wdXRzIiwgc2lkZV9lZmZlY3Q9bGFtYmRhIF9wLCBfbSwgX2MsIGV4LCBfeCwgX3M6IGV4KSwKICAgICAgICAgICAgcGF0Y2gub2JqZWN0KAogICAgICAgICAgICAgICAgdHJhaW5fbW9kdWxlLCAiX2JhdGNoX2Fuc3dlcl9sb2dwcm9icyIsCiAgICAgICAgICAgICAgICBzaWRlX2VmZmVjdD1sYW1iZGEgX20sIGV4LCBfZiwgX3ksIF9uOiBbKF9TY29yZVZhbHVlKDAuMCksIF9TY29yZVZhbHVlKDAuMCkpIGZvciBfIGluIGV4XSwKICAgICAgICAgICAgKSwKICAgICAgICApOgogICAgICAgICAgICByZXN1bHQgPSBfc2NvcmVfcm93cygKICAgICAgICAgICAgICAgIFt7InVzZXJfaWQiOiAidSIsICJoaXN0b3J5IjogWzFdLCAidGFyZ2V0IjogMywgImNhbmRpZGF0ZXMiOiBbOSwgMywgN119XSwKICAgICAgICAgICAgICAgIG9iamVjdCgpLCBtb2RlbCwge30sIDgxOTIsIDEsIDAsIF9TY29yZVRvcmNoKCksIG9iamVjdCgpLCBOb25lLAogICAgICAgICAgICApCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbMF1bInJhbmtlZF9jYW5kaWRhdGVzIl0sIFs5LCAzLCA3XSkKCiAgICBkZWYgdGVzdF9zZnRfdGFza19pc19iYWxhbmNlZF95ZXNfbm9fd2l0aF9zZWVkZWRfbmVnYXRpdmVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcGFpcnMgPSBbCiAgICAgICAgICAgIHsiaGlzdG9yeSI6IFsxLCAyLCAzXSwgInBvc2l0aXZlIjogMTAgKyBpLCAic2Z0X25lZ2F0aXZlIjogMTAwICsgaSwgIm5lZ2F0aXZlIjogMjAwICsgaX0KICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UoNikKICAgICAgICBdCiAgICAgICAgZXhhbXBsZXMgPSBfYnVpbGRfc2Z0X2V4YW1wbGVzKHBhaXJzLCA3LCA4LCBzZWVkPTIwMjQpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChleGFtcGxlcywgX2J1aWxkX3NmdF9leGFtcGxlcyhwYWlycywgNywgOCwgc2VlZD0yMDI0KSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bShlWyJhbnN3ZXJfdG9rZW4iXSA9PSA3IGZvciBlIGluIGV4YW1wbGVzKSwgNikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bShlWyJhbnN3ZXJfdG9rZW4iXSA9PSA4IGZvciBlIGluIGV4YW1wbGVzKSwgNikKICAgICAgICBmb3IgZXhhbXBsZSBpbiBleGFtcGxlczoKICAgICAgICAgICAgZXhwZWN0ZWQgPSB7NzogezEwICsgaSBmb3IgaSBpbiByYW5nZSg2KX0sIDg6IHsxMDAgKyBpIGZvciBpIGluIHJhbmdlKDYpfX0KICAgICAgICAgICAgc2VsZi5hc3NlcnRJbihleGFtcGxlWyJjYW5kaWRhdGUiXSwgZXhwZWN0ZWRbZXhhbXBsZVsiYW5zd2VyX3Rva2VuIl1dKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoYW55KGVbImNhbmRpZGF0ZSJdID49IDIwMCBmb3IgZSBpbiBleGFtcGxlcyksICJoYXJkIG5lZ2F0aXZlcyBhcmUgRFBPLW9ubHkiKQoKICAgIGRlZiB0ZXN0X2Ryb3BvdXRfaXNfemVyb2VkX2luY2x1ZGluZ19sb3JhX2Ryb3BvdXQoc2VsZikgLT4gTm9uZToKICAgICAgICBjbGFzcyBEcm9wb3V0OgogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgcCk6CiAgICAgICAgICAgICAgICBzZWxmLnAgPSBwCgogICAgICAgIGNsYXNzIE1vZGVsOgogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kdWxlcyk6CiAgICAgICAgICAgICAgICBzZWxmLl9tb2R1bGVzID0gbW9kdWxlcwoKICAgICAgICAgICAgZGVmIG1vZHVsZXMoc2VsZik6CiAgICAgICAgICAgICAgICByZXR1cm4gaXRlcihzZWxmLl9tb2R1bGVzKQoKICAgICAgICBjbGFzcyBObjoKICAgICAgICAgICAgcGFzcwoKICAgICAgICBjbGFzcyBUb3JjaDoKICAgICAgICAgICAgbm4gPSBObigpCgogICAgICAgIFRvcmNoLm5uLkRyb3BvdXQgPSBEcm9wb3V0CiAgICAgICAgbW9kdWxlcyA9IFtEcm9wb3V0KDAuMDUpLCBEcm9wb3V0KDAuMCksIG9iamVjdCgpLCBEcm9wb3V0KDAuMSldCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChfZGlzYWJsZV9kcm9wb3V0KE1vZGVsKG1vZHVsZXMpLCBUb3JjaCksIDIpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbChtLnAgPT0gMC4wIGZvciBtIGluIG1vZHVsZXMgaWYgaXNpbnN0YW5jZShtLCBEcm9wb3V0KSkpCgogICAgZGVmIHRlc3RfbG9yYV9zY29wZV9pc19sYW5ndWFnZV9sYXllcnNfb25seShzZWxmKSAtPiBOb25lOgogICAgICAgIGltcG9ydCByZQoKICAgICAgICBsYW5ndWFnZSA9IFsKICAgICAgICAgICAgImJhc2VfbW9kZWwubW9kZWwubW9kZWwubGF5ZXJzLjAuc2VsZl9hdHRuLnFfcHJvaiIsCiAgICAgICAgICAgICJiYXNlX21vZGVsLm1vZGVsLm1vZGVsLmxheWVycy4zNS5tbHAuZG93bl9wcm9qIiwKICAgICAgICBdCiAgICAgICAgZnJvemVuID0gWwogICAgICAgICAgICAiYmFzZV9tb2RlbC5tb2RlbC52aXN1YWwuYmxvY2tzLjMubWxwLmdhdGVfcHJvaiIsCiAgICAgICAgICAgICJiYXNlX21vZGVsLm1vZGVsLnZpc3VhbC5ibG9ja3MuMy5hdHRuLnFrdiIsCiAgICAgICAgICAgICJiYXNlX21vZGVsLm1vZGVsLnZpc3VhbC5tZXJnZXIubWxwLjAiLAogICAgICAgICAgICAiYmFzZV9tb2RlbC5tb2RlbC5sbV9oZWFkIiwKICAgICAgICBdCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFsbChyZS5mdWxsbWF0Y2goTE9SQV9UQVJHRVRfUkVHRVgsIG5hbWUpIGZvciBuYW1lIGluIGxhbmd1YWdlKSkKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKGFueShyZS5mdWxsbWF0Y2goTE9SQV9UQVJHRVRfUkVHRVgsIG5hbWUpIGZvciBuYW1lIGluIGZyb3plbikpCgogICAgZGVmIHRlc3RfanNvbl9jb252ZXJ0ZXJfaGFuZGxlc19zY2FsYXJfcHJvdG9jb2xfYW5kX3JlamVjdHNfdW5rbm93bl92YWx1ZXMoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmFzc2VydEFsbW9zdEVxdWFsKF9qc29uX2RlZmF1bHQoX1NjYWxhcigpKSwgMS41KQogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVHlwZUVycm9yKToKICAgICAgICAgICAgX2pzb25fZGVmYXVsdChvYmplY3QoKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==", "test_hanorec_protocol.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHVuaXR0ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKZnJvbSBhdWRpdCBpbXBvcnQgYXVkaXRfYXJ0aWZhY3RzLCBwcm90b2NvbF9lcnJvcnMgICMgbm9xYTogRTQwMgpmcm9tIHByZXAgaW1wb3J0IF9idWlsZF9mcmVxdWVuY3lfbWF0Y2hlZF9kZXJhbmdlbWVudCwgX3NlZWRlZF9zZWxlY3Rpb24gICMgbm9xYTogRTQwMgoKUFJPVE9DT0wgPSBST09UIC8gInJ1bi1wcm90b2NvbC5qc29uIgoKCmNsYXNzIEhhTm9SZWNQcm90b2NvbFRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3Byb3RvY29sX2lzX2NvbXBsZXRlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcHJvdG9jb2wgPSBqc29uLmxvYWRzKFBST1RPQ09MLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb3RvY29sX2Vycm9ycyhwcm90b2NvbCksIFtdKQoKICAgIGRlZiB0ZXN0X3NlbGVjdGlvbl9pc19zZWVkZWRfYW5kX25vdF9wcmVmaXhfYmlhc2VkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm93cyA9IFsoaW5kZXgsIFtpbmRleCwgaW5kZXggKyAxLCBpbmRleCArIDIsIGluZGV4ICsgM10pIGZvciBpbmRleCBpbiByYW5nZSgyMCldCiAgICAgICAgZmlyc3QgPSBfc2VlZGVkX3NlbGVjdGlvbihyb3dzLCA1LCAyMDI0LCAidHJhaW4iKQogICAgICAgIHNlY29uZCA9IF9zZWVkZWRfc2VsZWN0aW9uKHJvd3MsIDUsIDIwMjQsICJ0cmFpbiIpCiAgICAgICAgb3RoZXIgPSBfc2VlZGVkX3NlbGVjdGlvbihyb3dzLCA1LCAyMDI1LCAidHJhaW4iKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmlyc3QsIHNlY29uZCkKICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKGZpcnN0LCBvdGhlcikKICAgICAgICBzZWxmLmFzc2VydE5vdEVxdWFsKFtyb3dbMF0gZm9yIHJvdyBpbiBmaXJzdF0sIGxpc3QocmFuZ2UoNSkpKQoKICAgIGRlZiB0ZXN0X2ZyZXF1ZW5jeV9zaHVmZmxlX2lzX2JpamVjdGl2ZV9hbmRfZml4ZWRfcG9pbnRfZnJlZShzZWxmKSAtPiBOb25lOgogICAgICAgIGl0ZW1faWRzID0gbGlzdChyYW5nZSgxLCA5KSkKICAgICAgICBmcmVxdWVuY2llcyA9IHsxOiA1LCAyOiA1LCAzOiA0LCA0OiA0LCA1OiAzLCA2OiAzLCA3OiAyLCA4OiAyfQogICAgICAgIG1hcHBpbmcgPSBfYnVpbGRfZnJlcXVlbmN5X21hdGNoZWRfZGVyYW5nZW1lbnQoaXRlbV9pZHMsIGZyZXF1ZW5jaWVzLCAyMDI0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KG1hcHBpbmcpLCBzZXQoaXRlbV9pZHMpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2V0KG1hcHBpbmcudmFsdWVzKCkpLCBzZXQoaXRlbV9pZHMpKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwoc291cmNlICE9IHRhcmdldCBmb3Igc291cmNlLCB0YXJnZXQgaW4gbWFwcGluZy5pdGVtcygpKSkKCiAgICBkZWYgdGVzdF9hcnRpZmFjdF9hdWRpdF9yZWplY3RzX3plcm9fYXJtcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHByb3RvY29sID0ganNvbi5sb2FkcyhQUk9UT0NPTC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0ZW1wb3Jhcnk6CiAgICAgICAgICAgIHJvb3QgPSBQYXRoKHRlbXBvcmFyeSkKICAgICAgICAgICAgKHJvb3QgLyAic2Z0X2J1bmRsZS5qc29uIikud3JpdGVfdGV4dCgie30iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICAocm9vdCAvICJzZnRfbG9yYV9zdGF0ZS5wdCIpLndyaXRlX2J5dGVzKGIic3RhdGUiKQogICAgICAgICAgICByZXN1bHQgPSBhdWRpdF9hcnRpZmFjdHMocm9vdCwgcHJvdG9jb2wpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0WyJzdGF0dXMiXSwgIkZBSUwiKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoYW55KCJhcm0gbWF0cml4IG1pc21hdGNoIiBpbiBlcnJvciBmb3IgZXJyb3IgaW4gcmVzdWx0WyJlcnJvcnMiXSkpCgogICAgZGVmIHRlc3Rfc21va2VfYXVkaXRfcmVqZWN0c19taXNzaW5nX2NvbXB1dGVfdGVsZW1ldHJ5KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcHJvdG9jb2wgPSBqc29uLmxvYWRzKFBST1RPQ09MLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRlbXBvcmFyeToKICAgICAgICAgICAgcm9vdCA9IFBhdGgodGVtcG9yYXJ5KQogICAgICAgICAgICAocm9vdCAvICJzZnRfYnVuZGxlLmpzb24iKS53cml0ZV90ZXh0KCJ7fSIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIChyb290IC8gInNmdF9sb3JhX3N0YXRlLnB0Iikud3JpdGVfYnl0ZXMoYiJzdGF0ZSIpCiAgICAgICAgICAgIChyb290IC8gImFybS5wdCIpLndyaXRlX2J5dGVzKGIiY2hlY2twb2ludCIpCiAgICAgICAgICAgIChyb290IC8gImFybV9yZXN1bHQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHsKICAgICAgICAgICAgICAgICAgICAid2VpZ2h0IjogMS4wLAogICAgICAgICAgICAgICAgICAgICJpbWFnZV9jb25kaXRpb24iOiAicmVhbCIsCiAgICAgICAgICAgICAgICAgICAgInN0YXR1cyI6ICJDT01QTEVURSIsCiAgICAgICAgICAgICAgICAgICAgInByZWRpY3Rpb25zIjogW3sKICAgICAgICAgICAgICAgICAgICAgICAgIm5kY2dAMTAiOiAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICJyZWNhbGxAMTAiOiAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICJjYW5kaWRhdGVfcmVjYWxsQDIwIjogMC4wLAogICAgICAgICAgICAgICAgICAgIH1dLAogICAgICAgICAgICAgICAgICAgICJjaGVja3BvaW50IjogImFybS5wdCIsCiAgICAgICAgICAgICAgICAgICAgInBhcmVudF9zZnRfc2hhMjU2IjogImhhc2giLAogICAgICAgICAgICAgICAgfSksCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIChyb290IC8gInNtb2tlX21hbmlmZXN0Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAganNvbi5kdW1wcyh7CiAgICAgICAgICAgICAgICAgICAgInB1cnBvc2UiOiAiY29ycmVjdG5lc3Nfb25seV9ub25fc2lnbmFsIiwKICAgICAgICAgICAgICAgICAgICAibXVzdF9ub3RfYmVfdXNlZF9hc19zaWduYWwiOiBUcnVlLAogICAgICAgICAgICAgICAgfSksCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlc3VsdCA9IGF1ZGl0X2FydGlmYWN0cyhyb290LCBwcm90b2NvbCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbInN0YXR1cyJdLCAiRkFJTCIpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oImludmFsaWQgc2Z0X2dwdV9tZW1vcnkgdGVsZW1ldHJ5IiwgcmVzdWx0WyJlcnJvcnMiXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRJbigiaW52YWxpZCBkcG9fc3RhZ2VfdGltaW5ncyBvcHRpbWl6ZXIgdGltaW5nIiwgcmVzdWx0WyJlcnJvcnMiXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==", "test_hanorec_analysis.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQppbXBvcnQgc3lzCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCkpCmZyb20gYW5hbHlzaXMgaW1wb3J0IGZ1bGxfY29ob3J0X3ZhbHVlcywgaG9sbV9hZGp1c3QsIHBhaXJlZF9ib290c3RyYXAsIHJldHJpZXZlcl92ZXJkaWN0LCBzdW1tYXJpemUgICMgbm9xYTogRTQwMgoKCmRlZiBfcm93KHVzZXIsIHRhcmdldCwgbmRjZz0wLjAsIHJlY2FsbD0wLjAsIGNhbmQ9MS4wKToKICAgIHJldHVybiB7InVzZXJfaWQiOiB1c2VyLCAidGFyZ2V0IjogdGFyZ2V0LCAibmRjZ0AxMCI6IG5kY2csICJyZWNhbGxAMTAiOiByZWNhbGwsICJjYW5kaWRhdGVfcmVjYWxsQDIwIjogY2FuZH0KCgpkZWYgX3ZhbHMoZGVsdGFzKToKICAgIHJldHVybiBbeyJuZGNnQDEwIjogZCwgInJlY2FsbEAxMCI6IDAuMCwgImNhbmRpZGF0ZV9yZWNhbGxAMjAiOiAwLjB9IGZvciBkIGluIGRlbHRhc10KCgpjbGFzcyBBbmFseXNpc1Rlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3VuaW5mb3JtYXRpdmVfdXNlcnNfY291bnRfYXNfemVyb19pbl90aGVfZnVsbF9jb2hvcnQoc2VsZikgLT4gTm9uZToKICAgICAgICBjb2hvcnQgPSBbX3JvdygiYSIsIDEsIGNhbmQ9MC4wKSwgX3JvdygiYiIsIDIpLCBfcm93KCJjIiwgMywgY2FuZD0wLjApLCBfcm93KCJkIiwgNCwgY2FuZD0wLjApXQogICAgICAgIHZhbHVlcyA9IGZ1bGxfY29ob3J0X3ZhbHVlcyhjb2hvcnQsIFtfcm93KCJiIiwgMiwgbmRjZz0xLjAsIHJlY2FsbD0xLjApXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bW1hcml6ZSh2YWx1ZXMpWyJuZGNnQDEwIl0sIDAuMjUpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgZnVsbF9jb2hvcnRfdmFsdWVzKGNvaG9ydCwgW19yb3coInoiLCA5LCBuZGNnPTEuMCldKQoKICAgIGRlZiB0ZXN0X2Ryb3BwaW5nX2FuX2luZm9ybWF0aXZlX3VzZXJfaXNfYW5fZXJyb3Jfbm90X2Ffc2lsZW50X3plcm8oc2VsZikgLT4gTm9uZToKICAgICAgICBjb2hvcnQgPSBbX3JvdygiYSIsIDEpLCBfcm93KCJiIiwgMildCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgZnVsbF9jb2hvcnRfdmFsdWVzKGNvaG9ydCwgW19yb3coImIiLCAyLCBuZGNnPTEuMCldKQoKICAgIGRlZiB0ZXN0X2Jvb3RzdHJhcF9jaV9icmFja2V0c190aGVfdHJ1ZV9tZWFuX2FuZF9pc19kZXRlcm1pbmlzdGljKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbSg3KQogICAgICAgIGRlbHRhcyA9IFtybmcuY2hvaWNlKFstMC4yLCAwLjAsIDAuMCwgMC4zLCAwLjVdKSBmb3IgXyBpbiByYW5nZSgyMDApXQogICAgICAgIHRydWVfbWVhbiA9IHN1bShkZWx0YXMpIC8gbGVuKGRlbHRhcykKICAgICAgICByZXN1bHQgPSBwYWlyZWRfYm9vdHN0cmFwKF92YWxzKGRlbHRhcyksIF92YWxzKFswLjBdICogMjAwKSwgbj0yMDAwKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0LCBwYWlyZWRfYm9vdHN0cmFwKF92YWxzKGRlbHRhcyksIF92YWxzKFswLjBdICogMjAwKSwgbj0yMDAwKSkKICAgICAgICBsb3csIGhpZ2ggPSByZXN1bHRbImNpOTUiXQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyhsb3csIHRydWVfbWVhbikKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIoaGlnaCwgdHJ1ZV9tZWFuKQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyhoaWdoIC0gbG93LCAwLjE1KQogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwocmVzdWx0WyJtZWFuX2RlbHRhIl0sIHRydWVfbWVhbikKCiAgICBkZWYgdGVzdF9jb25zaXN0ZW50X2dhaW5faXNfZGV0ZWN0ZWRfYW5kX3BfdmFsdWVfaXNfbmV2ZXJfemVybyhzZWxmKSAtPiBOb25lOgogICAgICAgIHJlc3VsdCA9IHBhaXJlZF9ib290c3RyYXAoX3ZhbHMoWzAuNV0gKiA0MCksIF92YWxzKFswLjBdICogNDApLCBuPTUwMCkKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIocmVzdWx0WyJjaTk1Il1bMF0sIDApCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKHJlc3VsdFsicF90d29fc2lkZWQiXSwgMC4wKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmV0cmlldmVyX3ZlcmRpY3QocmVzdWx0KSwgIkJFQVRTX1JFVFJJRVZFUiIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbImNoYW5nZWRfdXNlcnMiXSwgNDApCgogICAgZGVmIHRlc3Rfbm9fZWZmZWN0X2NpX2NvbnRhaW5zX3plcm8oc2VsZikgLT4gTm9uZToKICAgICAgICBybmcgPSByYW5kb20uUmFuZG9tKDMpCiAgICAgICAgZGVsdGFzID0gW3JuZy5jaG9pY2UoWy0wLjMsIDAuM10pIGZvciBfIGluIHJhbmdlKDMwMCldCiAgICAgICAgcmVzdWx0ID0gcGFpcmVkX2Jvb3RzdHJhcChfdmFscyhkZWx0YXMpLCBfdmFscyhbMC4wXSAqIDMwMCksIG49MjAwMCkKICAgICAgICBzZWxmLmFzc2VydExlc3MocmVzdWx0WyJjaTk1Il1bMF0sIDAuMCkKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIocmVzdWx0WyJjaTk1Il1bMV0sIDAuMCkKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIocmVzdWx0WyJwX3R3b19zaWRlZCJdLCAwLjA1KQoKICAgIGRlZiB0ZXN0X3ZlcmRpY3RfZG9lc19ub3RfaGlkZV9kZWdyYWRhdGlvbl9ub2lzZV9vcl90cml2aWFsX2dhaW4oc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJldHJpZXZlcl92ZXJkaWN0KHsiY2k5NSI6IFstMC4wNSwgLTAuMDFdfSksICJXT1JTRV9USEFOX1JFVFJJRVZFUiIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXRyaWV2ZXJfdmVyZGljdCh7ImNpOTUiOiBbLTAuMDUsIDAuMDAxXX0pLCAiTk9fTUVBTklOR0ZVTF9HQUlOIikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJldHJpZXZlcl92ZXJkaWN0KHsiY2k5NSI6IFstMC4wMiwgMC4wM119KSwgIklOQ09OQ0xVU0lWRSIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXRyaWV2ZXJfdmVyZGljdCh7ImNpOTUiOiBbMC4wMDAxLCAwLjAwMDVdfSksICJHQUlOX0JFTE9XX01FQU5JTkdGVUwiKQoKICAgIGRlZiB0ZXN0X2hvbG1faXNfbW9ub3RvbmVfYW5kX2NhcHBlZChzZWxmKSAtPiBOb25lOgogICAgICAgIGFkanVzdGVkID0gaG9sbV9hZGp1c3QoeyJ4IjogMC4wMSwgInkiOiAwLjA0fSkKICAgICAgICBzZWxmLmFzc2VydEFsbW9zdEVxdWFsKGFkanVzdGVkWyJ4Il0sIDAuMDIpCiAgICAgICAgc2VsZi5hc3NlcnRBbG1vc3RFcXVhbChhZGp1c3RlZFsieSJdLCAwLjA0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoaG9sbV9hZGp1c3QoeyJ4IjogMC45LCAieSI6IDAuOTV9KSwgeyJ4IjogMS4wLCAieSI6IDEuMH0pCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo="}
SOURCE_SHA256 = {
 "prep.py": "36b31873cebc7521fb02db1d1629905d955a6ad14c5e5cbf68bd98b0907cf0b1",
 "train.py": "3e1d425e3c8f4f84cddcbc7055c53d3fa3393b814e442495cd0cbff14b890159",
 "analysis.py": "d89e9d1730baf7f7aa1e23b22e0a09dbeee13c69ac8538f5797f679d8c49eb6a",
 "experiment.json": "498662ade673a296e6501e44b5585f9914b58613d334a3a92f32af30ab9f3f09",
 "audit.py": "e48b89cbbd50d760724b308c3bc1773bf022740e67bc3c41d5436ab2954393c3",
 "run-protocol.json": "86f9c43e726084dcf5ac3ba20ae841513f8c38895e647867a36b4718dd10671a",
 "test_hanorec_fidelity.py": "fe2e99b8c37752792074b29561dec3bb8be8654357cb2308ca901f438a08ddc3",
 "test_hanorec_protocol.py": "f7ba56e9091420c26dea5181672d2f3e470ddd771d8c88f35daf4617ae76cd29",
 "test_hanorec_analysis.py": "dd715c36b6a1cd2552924e484327a564acbc89b6d07770c1454a3d9e74cf25b5"
}
import base64
SOURCE_DIR = Path("/kaggle/working/v11_source")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for name, blob in EMBEDDED_SOURCES.items():
    data = base64.b64decode(blob)
    assert hashlib.sha256(data).hexdigest() == SOURCE_SHA256[name], f"embedded source corrupted: {name}"
    (SOURCE_DIR / name).write_bytes(data)
sys.path.insert(0, str(SOURCE_DIR))
print({name: SOURCE_SHA256[name][:12] for name in SOURCE_SHA256})

## 3. CPU regression tests

In [ ]:
# CPU regression tests for every repaired behavior (SFT task, dropout, LoRA scope, tie order, statistics).
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(SOURCE_DIR), "-p", "test_hanorec_*.py"],
               cwd=SOURCE_DIR, check=True)

## 4. Protocol configuration

## 5. Locate and verify the SFT parent
Mounted from the SFT kernel's output. Source hashes, `sft_bundle.json` and adapter hash must match.

In [ ]:
import copy
import train as T
import analysis as A
INPUT = Path("/kaggle/input")
manifests = sorted(INPUT.rglob("job_manifest.json"))
assert len(manifests) == 1, f"expected exactly one SFT job_manifest.json, got {manifests}"
OUT_SFT = manifests[0].parent
PARENT = json.loads(manifests[0].read_text())
assert PARENT["job"] == "sft" and PARENT["status"] == "COMPLETE"
assert PARENT["sft_gate_passed"], ("SFT reference did not learn the Yes/No task "
                                   f"({PARENT['summary']['gate']}); fix SFT before spending DPO hours")
for name in ("prep.py", "train.py", "analysis.py", "experiment.json"):
    assert PARENT["source_sha256"][name] == SOURCE_SHA256[name], f"source drift vs SFT job: {name}"
SFT_DIR = OUT_SFT / "sft_real"
assert hashlib.sha256((SFT_DIR / "sft_bundle.json").read_bytes()).hexdigest() == PARENT["sft_bundle_sha256"]
assert hashlib.sha256((SFT_DIR / "sft_lora_state.pt").read_bytes()).hexdigest() == PARENT["sft_state_sha256"]
CONFIG = {**PARENT["config"], "artifact_pins": json.loads((SOURCE_DIR / "experiment.json").read_text())["artifact_pins"]}
T_EX = PARENT["seconds_per_example"]
SESSION_LIMIT_S = 5.6 * 3600     # MCP push timeout cap is 6 h; keep a margin for statistics/packaging
N_ARMS = 4
per_arm_s = (SESSION_LIMIT_S - 1200) / N_ARMS - 900          # setup + per-arm model load/scoring overhead
CONFIG["dpo_epochs"] = int(max(1, min(5, per_arm_s // (2 * CONFIG["train_pairs"] * T_EX * 1.4))))
ADAPTATIONS = {**PARENT["adaptations"],
               "dpo_epochs": None if CONFIG["dpo_epochs"] == 5 else {"frozen": 5, "used": CONFIG["dpo_epochs"],
                                                                      "reason": f"6 h session cap, {N_ARMS} arms, {T_EX:.2f}s/example"}}
print({"dpo_epochs": CONFIG["dpo_epochs"], "adaptations": ADAPTATIONS})
BUNDLE = json.loads((SFT_DIR / "sft_bundle.json").read_text())
COHORT = BUNDLE["baselines"]["sasrec_validation"]
print({"max_pixels": CONFIG["max_pixels"], "informative": len(BUNDLE["validation"]), "cohort": len(COHORT),
       "sft_gate": PARENT["summary"]["gate"]["reference_learned_binary_task"]})
OUT_ROOT = Path("/kaggle/working/v11_arms"); OUT_ROOT.mkdir(parents=True, exist_ok=True)
ARM_RESULTS = {}
ARM_SECONDS = []

## 6. Arms (one block per arm)
Each arm: fresh DPO adapter over the frozen SFT adapter, SFT-only reference, dropout off, NoDO noise,
epochs from measured speed (<= 5, identical across arms), micro-batch 4 x accumulation 8. Fail-closed checks inside `run_arm`: policy == reference at
step 0 (`init_parity_max_abs_logit`), exact optimizer-update count, SFT adapter unchanged, checkpoint reload.
A block is skipped (and recorded) if it cannot finish inside the session limit.

In [ ]:
def run_one(weight, control):
    tag = f"w{weight}" + (f"_{control}" if control else "")
    if tag in ARM_RESULTS:
        return ARM_RESULTS[tag]
    eta = max(ARM_SECONDS) * 1.15 if ARM_SECONDS else 0.0
    if time.monotonic() - NOTEBOOK_T0 + eta > SESSION_LIMIT_S:
        ARM_RESULTS[tag] = {"status": "SKIPPED_SESSION_BUDGET"}
        print(tag, ARM_RESULTS[tag]); return ARM_RESULTS[tag]
    t0 = time.monotonic()
    arm = T.run_arm({**CONFIG, "hardness_control": control}, SFT_DIR, float(weight), "real",
                    OUT_ROOT / f"arm_{tag}", SOURCE_DIR)
    ARM_SECONDS.append(time.monotonic() - t0)
    assert arm["status"] == "COMPLETE" and arm["predictions"] == [], "test scoring must stay disabled"
    ARM_RESULTS[tag] = arm
    losses = arm["losses"]
    print({"arm": tag, "val_ndcg@10_informative": arm["validation_mean_ndcg@10"],
           "init_parity": arm["init_parity_max_abs_logit"], "dropout_off": arm["dropout_modules_disabled"],
           "first_loss": round(losses[0], 4), "mean_first_20": round(sum(losses[:20]) / len(losses[:20]), 4),
           "mean_last_20": round(sum(losses[-20:]) / len(losses[-20:]), 4), "minutes": round(ARM_SECONDS[-1] / 60, 1)})
    return arm

### Arm w=1.0

In [ ]:
run_one(1.0, None)

### Arm w=0.0

In [ ]:
run_one(0.0, None)

### Arm w=0.5

In [ ]:
run_one(0.5, None)

### Arm w=0.0, constant-hardness control (mean)

In [ ]:
run_one(0.0, 'mean')

## 7. Paired statistics (validation only)
Planned primary contrasts: w=0.0 vs w=1.0 and w=0.5 vs w=1.0 (NDCG@10, Holm-adjusted). Every arm is also
compared with the SASRec order and the SFT-only reranker so degradation is never hidden.

In [ ]:
vals = {"sasrec": A.full_cohort_values(COHORT, COHORT),
        "sft_only": A.full_cohort_values(COHORT, BUNDLE["sft_validation_predictions"])}
for tag, arm in ARM_RESULTS.items():
    if arm.get("status") == "COMPLETE":
        vals[tag] = A.full_cohort_values(COHORT, arm["validation_predictions"])
RESULT = {"metrics": {k: A.summarize(v) for k, v in vals.items()}, "vs_sasrec": {}, "vs_sft_only": {}, "mechanism": {}}
for tag in vals:
    if tag in ("sasrec", "sft_only"):
        continue
    RESULT["vs_sasrec"][tag] = {m: A.paired_bootstrap(vals[tag], vals["sasrec"], m) for m in A.METRICS}
    RESULT["vs_sasrec"][tag]["verdict"] = A.retriever_verdict(RESULT["vs_sasrec"][tag]["ndcg@10"])
    RESULT["vs_sft_only"][tag] = A.paired_bootstrap(vals[tag], vals["sft_only"], "ndcg@10")
planned = {}
for tag in ("w0.0", "w0.5"):
    if tag in vals and "w1.0" in vals:
        planned[f"{tag}_minus_w1.0"] = A.paired_bootstrap(vals[tag], vals["w1.0"], "ndcg@10")
adjusted = A.holm_adjust({k: v["p_two_sided"] for k, v in planned.items()}) if planned else {}
RESULT["mechanism"] = {"contrasts": planned, "holm_adjusted_p": adjusted,
                       "min_meaningful_ndcg_delta": A.MIN_MEANINGFUL_NDCG_DELTA}
if "w0.0_mean" in vals and "w0.0" in vals:
    RESULT["mechanism"]["cf_signal_beyond_constant_hardness"] = A.paired_bootstrap(vals["w0.0"], vals["w0.0_mean"], "ndcg@10")
print(json.dumps({"metrics": RESULT["metrics"], "verdicts": {k: v["verdict"] for k, v in RESULT["vs_sasrec"].items()}}, indent=1))
print(json.dumps(RESULT["mechanism"], indent=1))

## 8. Package artifacts

In [ ]:
summary = {
    "job": "arms",
    "status": ("FAILED" if any(a.get("status") not in ("COMPLETE", "SKIPPED_SESSION_BUDGET") for a in ARM_RESULTS.values())
               else "PARTIAL_SESSION_BUDGET" if any(a.get("status") == "SKIPPED_SESSION_BUDGET" for a in ARM_RESULTS.values())
               else "COMPLETE"),
    "adaptations": ADAPTATIONS,
    "must_not_be_used_as_confirmation": True,
    "single_seed": CONFIG["seed"],
    "source_sha256": SOURCE_SHA256, "gpu": GPU, "parent_manifest_sha256": hashlib.sha256(manifests[0].read_bytes()).hexdigest(),
    "arms": {tag: {k: a.get(k) for k in ("status", "optimizer_updates", "expected_optimizer_updates", "init_parity_max_abs_logit",
                                         "dropout_modules_disabled", "hardness_control", "validation_mean_ndcg@10", "elapsed_seconds", "parent_sft_sha256")}
             for tag, a in ARM_RESULTS.items()},
    "result": RESULT, "elapsed_minutes": round((time.monotonic() - NOTEBOOK_T0) / 60, 1),
}
(OUT_ROOT / "arms_summary.json").write_text(json.dumps(summary, indent=2, default=str))
# Per-user predictions stay in each arm_result.json for later paired analysis; drop nothing.
print("PACKAGED", summary["status"], summary["elapsed_minutes"], "min")